# MOMENT-1-base — DIMER E2E supervised adaptation tutorial: activity classification from motion windows, linear probe vs bounded unfreeze (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/moment-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/moment-pipeline/blob/main/tutorials/moment_classification_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-AutonLab%2FMOMENT--1--base-ffcc4d?style=flat)](https://huggingface.co/AutonLab/MOMENT-1-base) [![Upstream](https://img.shields.io/badge/Upstream-moment--timeseries--foundation--model%2Fmoment-181717?style=flat&logo=github&logoColor=white)](https://github.com/moment-timeseries-foundation-model/moment) [![arXiv](https://img.shields.io/badge/arXiv-2402.03885-b31b1b.svg)](https://arxiv.org/abs/2402.03885)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** pretrained time-series representation extraction (pooled 768-d window embeddings) and bounded supervised adaptation to a labelled-window classification task — a linear probe on the frozen embeddings with an optional unfreeze of the last encoder blocks — measured by held-out accuracy and macro-F1, using the pinned `AutonLab/MOMENT-1-base` encoder

**This notebook is standalone.** It carries the repository's package (12 modules under `src/moment_pipeline/`, at revision `de30be9dca6c`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `9fea447e740eb968a9e8d80c7562ae122bdb5dde` (~454 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies (including `momentfm` from its pinned source commit), stages and digest-verifies the pinned MOMENT-1-base snapshot (safetensors, 454 MB), fetches the 79.6 MB UCI HAPT archive (no credential), cuts 179 digest-pinned ten-second motion windows from it and draws 107 / 36 / 36 training, validation and test windows by a seeded split of whole volunteers, runs three test windows through the inference contract — the public validation, windowing and embedding path with an input manifest, a rejection probe and the `not-measurable` per-batch report — scores the frozen embeddings on the test split by a majority floor, a cosine 5-NN vote and a linear probe (the **frozen policy**), trains a bounded unfreeze of the last two encoder blocks with the head and selects between it and the probe by validation log-loss (the **unfrozen policy**), scores the held-out split with the selected model, prints predictions before and after, exports the head and any trained blocks as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path takes about five minutes of model time after the downloads; a CUDA runtime is used automatically when present.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own labelled windows as a `.zip` holding `records.csv` (columns `id`, `file`, `label`, optional `group`) beside `.npy` arrays of shape `(channels, 512)` — arrays are decoded from the archive, never extracted to disk. They pass through the same validation, seeded group-disjoint split, floors, frozen-policy probe, unfrozen-policy training and selection, held-out evaluation, prediction, artifact export and reload-parity cells as the HAPT sample. The expected schema and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

At inference every 512-step window (each channel independently, 64 non-overlapping patches of 8 steps, instance-normalised per window) passes through the 12-block T5 encoder of MOMENT-1-base and the patch representations are mean-pooled into one 768-d vector per window; the `embedding` task instance replaces the pretrained reconstruction head with an identity, so the vector is a representation, not a prediction. The carried package adds manifest verification, the long-format validation and windowing contract, input validation with named ceilings, and the `validate_inputs`, `evaluation_report` and `build_provenance` helpers; the per-batch `evaluation_report` of an embedding is always `not-measurable`, because a vector has no intrinsic accuracy.

What this notebook adds to inference is **supervised adaptation under an explicit frozen-vs-unfrozen policy** — the first task-head contract of this repository, and a caller-trained one: MOMENT-1-base ships no classification head and its upstream classification head is not pretrained. The dataset is real: 179 ten-second windows of smartphone motion data from the UCI *Smartphone-Based Recognition of Human Activities and Postural Transitions* dataset (HAPT, CC BY 4.0) — for each of 30 volunteers and each of six activities (walking, walking upstairs, walking downstairs, sitting, standing, laying) the centre 512 samples of the first labelled segment long enough to hold them, over six channels (accelerometer and gyroscope x/y/z); the 79.6 MB archive is pinned by byte size and SHA-256, fetched from the UCI repository at run time and read without extraction. Windows of one volunteer share a body and a phone, so the sample is split by **volunteer**, never by window. The carried `adaptation.py` scores predictions by **accuracy** and **macro-F1** with per-class recall and a confusion matrix; a **majority floor** and a **cosine 5-NN vote** over the frozen embeddings frame the numbers. The **frozen policy** trains only a linear head on the frozen embeddings (a linear probe); the **unfrozen policy** continues from that probe by training the last encoder blocks with the head end to end, and the epoch with the lowest validation log-loss — which may be the probe itself — is kept. The adaptation question is whether unfreezing buys anything over the probe on 107 training windows; the representation question underneath it is what per-window instance normalisation does to activities that differ mainly by gravity's direction. Nothing here is a quality claim about your series: it is one seeded split of one small corpus.

**Learning objectives:** install the pinned runtime; read what the carried package guarantees; stage and digest-verify the immutable model revision; fetch a digest-pinned real labelled corpus and validate and split it by volunteer without leakage; push real windows through the public validation, windowing and embedding path and read the input manifest, the vector contract and the `not-measurable` report correctly; read accuracy and macro-F1 beside a majority floor and a k-NN baseline; train a linear probe on frozen embeddings and a bounded unfreeze with explicit hyperparameters and validation-based selection between the two policies; evaluate on an independent volunteer-disjoint test split; compare predictions before and after; and export a safetensors adapter (head plus any trained blocks) that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** forecasting, anomaly decisions, imputation quality claims, full-encoder or patch-embedding training, data augmentation, any pretrained classification head, and any claim that six activities from 30 volunteers stand in for your series. The repository exposes none of these; the adapted encoder still emits representations and the trained head answers only for its six labels.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; the public API accepts `float32` only. The build record measured about 0.06 s per six-channel window to embed on CPU (16 s for the 179-window k-NN pass) and about 0.85 s per window per training step on the last two encoder blocks, so a three-epoch unfreeze over 107 windows with four validation passes took about 150 s. The pinned `torch==2.14.0` install and the 454 MB checkpoint are the large downloads of the run, then the 79.6 MB archive.
- **Knowledge:** basic Python, NumPy and pandas; what a long-format time-series table is; what a linear probe is and why it is the cheapest honest test of a representation; what accuracy and macro-F1 measure; what validation-based selection between two policies means.
- **Data contract:** records are `{{id, x, label}}` — a float32 `(channels, 512)` array (or a path to a `.npy`) with 1..32 channels, finite values of magnitude at most 1,000, a label of 1..64 plain characters, ids matching `[A-Za-z0-9_.:-]{{1,64}}` and unique; a training set needs 8..1,024 records and 2..100 classes with one channel count throughout; windows are de-duplicated by sample digest and split by `user` / `group` so one person's data never straddles splits. Every window then passes through the package's long-format validation (`series_id, timestamp, channel, value`) and canonical windowing exactly as inference input does. BYOD accepts a `.zip` (or a directory) holding `records.csv` and the `.npy` files.
- **Validation is structural, not semantic:** nothing checks that a label is right for its window — a mislabelled set is trained on without complaint; the sample rate is assumed to be 50 Hz only for the timestamps the canonical path requires, and the model never sees it.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — wearable and motion recordings of identifiable people are exactly that. The default path uploads nothing.
- **External access (data):** besides the Hub snapshot, the default path fetches one pinned object — the HAPT archive at `https://archive.ics.uci.edu/static/public/341/smartphone+based+recognition+of+human+activities+and+postural+transitions.zip`, 79,596,192 bytes, SHA-256 `4ac4ae06…` in the carried `samples.py` — over HTTPS, refused on any byte-size or SHA-256 mismatch before it is opened; the dataset is CC BY 4.0 per the UCI repository's licence notice and is credited to its authors in the References.
- **External access:** the Hugging Face Hub only, to fetch the pinned `AutonLab/MOMENT-1-base` snapshot (~454 MB in total) at revision `9fea447e740e…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `pandas`, `numpy` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'momentfm @ git+https://github.com/moment-timeseries-foundation-model/moment@38f7310ad594100747ca2a8357e9c7ca7d323e0e',
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'numpy==2.5.3',
    'pandas==3.0.5',
    'huggingface-hub==1.30.0',
    'safetensors==0.8.0',
    'transformers==5.16.1',
]
NOTEBOOK_SOURCE = {
    'repository': 'moment-pipeline',
    'repository_revision': 'de30be9dca6c2f001cb91002cce7ed4ff00d220a',
    'embedded_module': 'src/moment_pipeline/model.py',
    'embedded_modules': ['src/moment_pipeline/config.py', 'src/moment_pipeline/model.py', 'src/moment_pipeline/samples.py', 'src/moment_pipeline/validation.py', 'src/moment_pipeline/canonical.py', 'src/moment_pipeline/csvio.py', 'src/moment_pipeline/embedding.py', 'src/moment_pipeline/provenance.py', 'src/moment_pipeline/adaptation.py', 'src/moment_pipeline/imputation.py', 'src/moment_pipeline/anomaly.py', 'src/moment_pipeline/roles.py'],
    'module_sha256': '33497e168bd1d5f2891cb3328187c62385ab84b5c93d16de5ed57111aefd90f1',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, pandas, numpy
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'pandas': pandas.__version__, 'numpy': numpy.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/moment_pipeline/` @ `de30be9dca6c`)

The next 12 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/12:** `src/moment_pipeline/config.py`

In [ ]:
"""Configuration for the pinned MOMENT-1-base pipeline.

Every value here is a DIMER-facing contract knob. The three structural constants
(`SEQUENCE_LENGTH`, `PATCH_LENGTH`, `PATCH_STRIDE`) mirror the pinned checkpoint's
`config.json` and are asserted against the downloaded file at load time
(`moment_pipeline.model.load_moment`) rather than trusted blindly.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Literal

#: Task names this pipeline exposes. v1 deliberately excludes ``forecasting`` and
#: ``classification``: those heads are freshly initialized, not pretrained (RFC M-1/M-4).
Task = Literal["embedding", "reconstruction"]

#: Structural constants of ``AutonLab/MOMENT-1-base`` at the pinned revision.
SEQUENCE_LENGTH = 512
PATCH_LENGTH = 8
PATCH_STRIDE = 8
N_PATCHES = SEQUENCE_LENGTH // PATCH_LENGTH  # 64

#: Finite sentinel substituted for missing payloads before the tensor reaches MOMENT.
#: Missingness itself is carried exclusively by the masks (RFC M-2, validation rules 11-12).
DEFAULT_PREFILL_VALUE = 0.0

#: Devices the pipeline accepts. ``auto`` resolves against the runtime; there is never a
#: hard CUDA requirement.
SUPPORTED_DEVICES = ("auto", "cpu", "cuda")

#: Dtypes the pipeline accepts. **float32 only in Phase 1.**
#:
#: ``float16`` and ``bfloat16`` were accepted here and by ``load_moment`` until they were
#: shown to be unreachable. ``load_moment`` casts the whole module with
#: ``pipeline.to(dtype=...)``, but ``embedding.embed`` and ``imputation.reconstruct`` build
#: their inputs from ``WindowSet.x_enc``, which ``canonical.to_windows`` constructs as
#: ``np.float32``, and carry only ``.to(device)``. A float32 activation entering a
#: half-precision ``nn.Linear`` raises ``RuntimeError: mat1 and mat2 must have the same
#: dtype`` on the first forward pass, so neither half dtype could ever have run. Nothing
#: caught it because no test exercised a non-float32 dtype.
#:
#: Narrowing rather than plumbing is deliberate: half precision on CPU is not uniformly
#: implemented in torch, and this repository does not expose a capability it has not
#: exercised against the real weights. Widening is a Phase-2 change and owes an
#: integration test per dtype, not just an input cast.
SUPPORTED_DTYPES = ("float32",)

#: One message for both entry points, so ``MomentConfig`` and ``load_moment`` cannot drift
#: into explaining the same refusal differently.
_UNSUPPORTED_DTYPE = (
    "unsupported dtype {dtype!r}; Phase 1 accepts float32 only. float16 and bfloat16 are "
    "refused because the inference paths feed MOMENT float32 tensors built from "
    "WindowSet.x_enc while load_moment casts the module, which raises "
    "'mat1 and mat2 must have the same dtype' on the first forward pass"
)


class ConfigError(ValueError):
    """Raised when a `MomentConfig` is internally inconsistent or unsupported."""


@dataclass(frozen=True)
class ResourceLimits:
    """Explicit resource guards (RFC common-validation rule 10)."""

    max_rows: int = 5_000_000
    max_series: int = 1024
    max_channels: int = 32
    max_windows: int = 1024

    def __post_init__(self) -> None:
        for name in ("max_rows", "max_series", "max_channels", "max_windows"):
            if getattr(self, name) < 1:
                raise ConfigError(f"{name} must be >= 1, got {getattr(self, name)}")


@dataclass(frozen=True)
class MomentConfig:
    """Runtime configuration for one MOMENT task instance.

    `task` is part of the identity of a loaded model: MOMENT replaces its head when
    the task changes (RFC M-8), so a config and a loaded pipeline travel together.
    """

    task: Task = "embedding"
    sequence_length: int = SEQUENCE_LENGTH
    patch_length: int = PATCH_LENGTH
    patch_stride: int = PATCH_STRIDE
    batch_size: int = 8
    device: str = "auto"
    dtype: str = "float32"
    prefill_value: float = DEFAULT_PREFILL_VALUE
    strict_frequency: bool = False
    limits: ResourceLimits = field(default_factory=ResourceLimits)

    def __post_init__(self) -> None:
        if self.task not in ("embedding", "reconstruction"):
            raise ConfigError(
                f"task must be 'embedding' or 'reconstruction' (v1 scope), got {self.task!r}"
            )
        if self.sequence_length != SEQUENCE_LENGTH:
            raise ConfigError(
                f"the pinned checkpoint is fixed at sequence_length={SEQUENCE_LENGTH}, "
                f"got {self.sequence_length}"
            )
        if self.patch_length != PATCH_LENGTH or self.patch_stride != PATCH_STRIDE:
            raise ConfigError(
                f"the pinned checkpoint is fixed at patch_len={PATCH_LENGTH} / "
                f"stride={PATCH_STRIDE}, got {self.patch_length}/{self.patch_stride}"
            )
        if self.batch_size < 1:
            raise ConfigError(f"batch_size must be >= 1, got {self.batch_size}")
        if self.device not in SUPPORTED_DEVICES:
            raise ConfigError(
                f"device must be one of {'|'.join(SUPPORTED_DEVICES)}, got {self.device!r}"
            )
        if self.dtype not in SUPPORTED_DTYPES:
            raise ConfigError(_UNSUPPORTED_DTYPE.format(dtype=self.dtype))
        import math

        if not math.isfinite(self.prefill_value):
            raise ConfigError(
                "prefill_value must be finite (that is the whole point), got "
                f"{self.prefill_value!r}"
            )

    @property
    def n_patches(self) -> int:
        return self.sequence_length // self.patch_length

    def resolved_device(self) -> str:
        """Resolve ``auto`` against the actual runtime. Never a hard CUDA requirement."""
        if self.device != "auto":
            return self.device
        try:
            import torch
        except ImportError:  # pragma: no cover - torch is a hard dependency in practice
            return "cpu"
        return "cuda" if torch.cuda.is_available() else "cpu"

**Module 2/12:** `src/moment_pipeline/model.py` (carried verbatim; see the note above)

In [ ]:
"""Pinned, integrity-verified loader for `AutonLab/MOMENT-1-base`.

The standard path is deliberately narrow (RFC "Supply-chain invariants"):

1. only `AutonLab/MOMENT-1-base` at revision `9fea447e…` is accepted — `main`, `latest`,
   any other sha, a local directory and any `s3://` / URL source are refused *before*
   any network call;
2. the resolved snapshot directory must be named after the pinned commit;
3. no pickle-format weight file may be present in the snapshot — `pytorch_model.bin`
   exists upstream at this very revision (453,978,525 bytes), so a silent `.bin` fallback
   is a live hazard, and the download uses `allow_patterns` so the file is never fetched.
   **These two exclusion controls are what guarantee which file was loaded** (RFC
   invariant 7); they are mutation-tested;
4. `config.json` and `model.safetensors` digests and the weight byte size are verified —
   on every load, including when the caller supplies its own `VerifiedSnapshot`, so
   provenance records what was verified rather than what was asserted;
5. after construction, every non-head tensor in the live module is compared byte-for-byte
   against the entries read straight out of `model.safetensors`. That proves the live
   *values* are the pinned checkpoint's — no fresh initialization, no partial or tampered
   load. It does **not** prove which file on disk was read: see (3).
"""

from __future__ import annotations

import hashlib
import os
from collections.abc import Callable
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from huggingface_hub import snapshot_download

# standalone rewrite (build_notebook.py): `from .config import (` removed — names are kernel globals defined by the carried modules

#: Fleet identity names (DIMER NOTEBOOK_SPEC 1.1 ST3): the constants every DIMER package spells
#: the same way, so a generated standalone notebook and the fleet tooling can read the identity
#: without knowing this package's own vocabulary. The ``PINNED_*`` names below are the package's
#: original spelling and stay in use everywhere; they alias these, never the reverse.
MODEL_ID = "AutonLab/MOMENT-1-base"
MODEL_REVISION = "9fea447e740eb968a9e8d80c7562ae122bdb5dde"
MODEL_KEY = "moment-1-base"

PINNED_MODEL_ID = MODEL_ID
PINNED_REVISION = MODEL_REVISION
PINNED_CONFIG_SHA256 = "f1c66c2bb845229c0ed27a1600dbcc956b85ab21f9e5fd8a1663e6641bed7755"
PINNED_CONFIG_BYTES = 949
PINNED_WEIGHTS_FILENAME = "model.safetensors"
PINNED_WEIGHTS_SHA256 = "1a436826ffe618273ec62b9656dc4cab8edc470364f104e90542a4ebc14fb825"
PINNED_WEIGHTS_BYTES = 453_940_120

#: Pickle-based weight formats. `pytorch_model.bin` is present upstream at the pinned
#: revision; the rest are the other serializations `torch.load` would happily unpickle, so
#: the guard is as wide as its own error message claims (R-11).
FORBIDDEN_WEIGHT_PATTERNS = ("*.bin", "*.pt", "*.pth", "*.ckpt", "*.pkl")
KNOWN_FORBIDDEN_WEIGHT_FILE = "pytorch_model.bin"
KNOWN_FORBIDDEN_WEIGHT_SHA256 = (
    "23c3d65bbb6dcd323352029e9fbe4ee3a3da0fff55b45ee4e00f38fff4e9bfb9"
)

ALLOW_PATTERNS = ["config.json", "model.safetensors", "README.md"]

#: Fleet snapshot scheme (DIMER NOTEBOOK_SPEC 1.1 MOD13): the pinned files also live in a
#: repository-local snapshot directory named by ``MODEL_KEY``, described by a committed
#: ``dimer-base-manifest.json`` (paths, byte sizes, SHA-256). The manifest is the parity anchor a
#: standalone notebook carries inline; ``verify_snapshot`` asserts its digests equal the
#: ``PINNED_*`` constants above on every load, so the two can never disagree silently.
MANIFEST_NAME = "dimer-base-manifest.json"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative

#: How the revision is established on this path, recorded in every export. The sibling
#: chronos-2 pipeline asks the Hub which commit the pin resolves to and records whether that
#: confirmation ran; this one does not, and says so rather than letting a bare `revision`
#: field imply a check that never happened.
REVISION_BASIS = (
    "established by content: config.json and model.safetensors hash to the pinned SHA-256 "
    "digests and the weight byte count matches. The snapshot directory name is compared "
    "against the pinned commit as a consistency assertion, but huggingface_hub names that "
    "directory after the requested revision, so it cannot fail for a SHA request. No "
    "independent Hub commit lookup is performed on this path"
)

#: The weights licence. There is NO LICENSE file in the HF repo at this revision; MIT is
#: declared in the model-card metadata only. Code licence is tracked separately (LICENSE).
MODEL_LICENSE = "MIT"
MODEL_LICENSE_BASIS = (
    "declared as `license: mit` in the model card metadata at revision "
    "9fea447e740eb968a9e8d80c7562ae122bdb5dde; no LICENSE file exists in the "
    "Hugging Face repository at that revision"
)

#: momentfm is installed from source at an exact upstream commit (RFC M-7).
MOMENTFM_SOURCE_URL = "https://github.com/moment-timeseries-foundation-model/moment"
MOMENTFM_SOURCE_COMMIT = "38f7310ad594100747ca2a8357e9c7ca7d323e0e"

_TASK_TO_UPSTREAM = {"embedding": "embedding", "reconstruction": "reconstruction"}

# The one validated configuration surface lives in `config` and is imported above, so
# `load_moment` and `MomentConfig.__post_init__` cannot drift apart (R-12). Phase 1
# accepts float32 only; `config.SUPPORTED_DTYPES` records why.


class ModelSourceError(ValueError):
    """The requested model source is not the single approved pinned source."""


class IntegrityError(RuntimeError):
    """A supply-chain assertion about the downloaded snapshot failed."""

    def __init__(self, code: str, message: str, details: dict[str, Any] | None = None):
        self.code = code
        self.details = details or {}
        super().__init__(f"[{code}] {message}")


@dataclass(frozen=True)
class VerifiedSnapshot:
    """A snapshot directory that has passed every supply-chain assertion."""

    path: Path
    model_id: str
    revision: str
    config_sha256: str
    config_bytes: int
    weights_path: Path
    weights_sha256: str
    weights_bytes: int
    seq_len: int
    patch_len: int
    patch_stride: int
    task_name_in_config: str


@dataclass(frozen=True)
class ModelIdentity:
    """Everything an export needs to say exactly what produced it."""

    name: str
    revision: str
    config_sha256: str
    weights_sha256: str
    weights_bytes: int
    license: str
    license_basis: str
    weight_file_loaded: str
    seq_len: int
    patch_len: int
    patch_stride: int
    d_model_effective: int
    task: str
    device: str
    dtype: str
    #: How `revision` was established. Never omitted: a bare revision string in an export
    #: reads as a verified commit, and on this path it is a content claim (see
    #: `REVISION_BASIS`), which is a different and narrower thing.
    revision_basis: str = REVISION_BASIS


@dataclass(eq=False)
class LoadedMoment:
    """A task-specific MOMENT instance plus its identity and load proof."""

    pipeline: Any = field(repr=False)
    identity: ModelIdentity
    snapshot: VerifiedSnapshot = field(repr=False)
    proof: dict[str, Any] = field(repr=False, default_factory=dict)


def sha256_file(path: str | os.PathLike[str], chunk: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()


def assert_pinned_source(model_id: str, revision: str) -> None:
    """Refuse anything but the one approved immutable source. No network is touched."""
    if not isinstance(model_id, str) or not isinstance(revision, str):
        raise ModelSourceError("model_id and revision must be strings")
    lowered = model_id.strip().lower()
    if "://" in lowered:
        raise ModelSourceError(
            f"URL-style model sources are refused in the standard path: {model_id!r}"
        )
    if lowered.startswith((".", "/", "~")) or "\\" in model_id or os.path.isabs(model_id):
        raise ModelSourceError(
            f"local-path model sources are refused in the standard path: {model_id!r}"
        )
    if model_id != PINNED_MODEL_ID:
        raise ModelSourceError(
            f"only {PINNED_MODEL_ID!r} is approved for the standard path, got {model_id!r}"
        )
    if revision != PINNED_REVISION:
        raise ModelSourceError(
            "the standard path resolves only the pinned immutable revision "
            f"{PINNED_REVISION!r}; refused {revision!r} "
            "(mutable refs such as 'main'/'latest' and other commits are not approved)"
        )


def find_forbidden_weight_files(directory: str | os.PathLike[str]) -> list[str]:
    root = Path(directory)
    found: list[str] = []
    for pattern in FORBIDDEN_WEIGHT_PATTERNS:
        found.extend(str(p.relative_to(root)) for p in root.rglob(pattern) if p.is_file())
    return sorted(set(found))


def _read_manifest(root: Path) -> dict[str, Any]:
    """Load and identity-check ``<root>/dimer-base-manifest.json``."""
    import json

    manifest_path = root / MANIFEST_NAME
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except (OSError, ValueError) as exc:
        raise IntegrityError(
            "MANIFEST_MISSING", f"could not read snapshot manifest {manifest_path}: {exc}"
        ) from exc
    if manifest.get("modelId") != PINNED_MODEL_ID or manifest.get("revision") != PINNED_REVISION:
        raise IntegrityError(
            "MANIFEST_IDENTITY_MISMATCH",
            f"snapshot manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {PINNED_MODEL_ID}@{PINNED_REVISION}",
            {"manifest": manifest_path.as_posix()},
        )
    if not isinstance(manifest.get("files"), list) or not manifest["files"]:
        raise IntegrityError("MANIFEST_EMPTY", f"snapshot manifest lists no files: {manifest_path}")
    return manifest


def has_manifest(directory: str | os.PathLike[str]) -> bool:
    """True when the directory is a fleet snapshot (``weights/<MODEL_KEY>/``) with a manifest."""
    return (Path(directory) / MANIFEST_NAME).is_file()


def verify_snapshot(path: str | os.PathLike[str] | None = None) -> dict[str, Any]:
    """Manifest-driven verification of a fleet snapshot directory; raise on the first mismatch.

    Every manifest entry is size- and SHA-256-checked, then the manifest's ``config.json`` and
    ``model.safetensors`` digests and the weight byte count are asserted **equal to** the pinned
    ``PINNED_*`` constants, so this path is never weaker than the revision-directory path of
    :func:`verify_snapshot_dir`. Returns ``{"path": ..., **manifest}``.
    """
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    if not root.is_dir():
        raise IntegrityError("SNAPSHOT_MISSING", f"snapshot directory not found: {root}")
    manifest = _read_manifest(root)
    forbidden = find_forbidden_weight_files(root)
    if forbidden:
        raise IntegrityError(
            "FORBIDDEN_WEIGHT_FILE",
            f"snapshot contains pickle weight file(s) {forbidden}; the standard path "
            "must load model.safetensors and must never risk a silent .bin fallback",
            {"files": forbidden},
        )
    digests: dict[str, str] = {}
    sizes: dict[str, int] = {}
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise IntegrityError(
                "SNAPSHOT_INCOMPLETE",
                f"manifest-listed file missing from snapshot: {entry['path']}",
            )
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise IntegrityError(
                "MANIFEST_SIZE_MISMATCH",
                f"{entry['path']} is {size} bytes, manifest says {entry['bytes']}",
                {"path": entry["path"], "actual": size, "expected": entry["bytes"]},
            )
        digest = sha256_file(file_path)
        if digest != entry["sha256"]:
            raise IntegrityError(
                "MANIFEST_DIGEST_MISMATCH",
                f"{entry['path']} sha256 {digest} != manifest {entry['sha256']}",
                {"path": entry["path"], "actual": digest, "expected": entry["sha256"]},
            )
        digests[entry["path"]] = digest
        sizes[entry["path"]] = size
    for required in ("config.json", PINNED_WEIGHTS_FILENAME):
        if required not in digests:
            raise IntegrityError(
                "MANIFEST_INCOMPLETE", f"snapshot manifest does not list {required} ({root})"
            )
    if digests["config.json"] != PINNED_CONFIG_SHA256:
        raise IntegrityError(
            "CONFIG_DIGEST_MISMATCH",
            f"manifest config.json sha256 {digests['config.json']} != "
            f"pinned {PINNED_CONFIG_SHA256}",
            {"actual": digests["config.json"], "expected": PINNED_CONFIG_SHA256},
        )
    if digests[PINNED_WEIGHTS_FILENAME] != PINNED_WEIGHTS_SHA256:
        raise IntegrityError(
            "WEIGHTS_DIGEST_MISMATCH",
            f"manifest {PINNED_WEIGHTS_FILENAME} sha256 {digests[PINNED_WEIGHTS_FILENAME]} != "
            f"pinned {PINNED_WEIGHTS_SHA256}",
            {"actual": digests[PINNED_WEIGHTS_FILENAME], "expected": PINNED_WEIGHTS_SHA256},
        )
    if sizes[PINNED_WEIGHTS_FILENAME] != PINNED_WEIGHTS_BYTES:
        raise IntegrityError(
            "WEIGHTS_SIZE_MISMATCH",
            f"manifest {PINNED_WEIGHTS_FILENAME} is {sizes[PINNED_WEIGHTS_FILENAME]} bytes, "
            f"pinned {PINNED_WEIGHTS_BYTES}",
            {"actual": sizes[PINNED_WEIGHTS_FILENAME], "expected": PINNED_WEIGHTS_BYTES},
        )
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at ``PINNED_REVISION`` straight into ``root``."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(PINNED_MODEL_ID, relative_path, revision=PINNED_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | os.PathLike[str] | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent from the fleet snapshot directory.

    A fresh clone commits the manifest and git-ignores the weights, so this is how the
    snapshot is populated. Only files named by the manifest are fetched, only at
    ``PINNED_REVISION``, and :func:`verify_snapshot` still re-hashes everything afterwards.
    Returns the relative paths fetched (empty when nothing was missing).
    """
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; pass allow_download=True to fetch them "
            f"at {PINNED_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def verify_snapshot_dir(
    directory: str | os.PathLike[str],
    *,
    expected_revision: str = PINNED_REVISION,
    expected_config_sha256: str = PINNED_CONFIG_SHA256,
    expected_weights_sha256: str = PINNED_WEIGHTS_SHA256,
    expected_weights_bytes: int = PINNED_WEIGHTS_BYTES,
    check_directory_name: bool = True,
) -> tuple[str, int, str, int, dict[str, Any]]:
    """Assert the snapshot is the pinned artifact. Returns digests plus the parsed config.

    Ordering matters: the forbidden-`.bin` check runs first, so a snapshot polluted with
    `pytorch_model.bin` is refused before anything else is considered.

    The `check_directory_name` comparison is a **consistency assertion, not an oracle**.
    `assert_pinned_source` has already forced the requested revision to be `PINNED_REVISION`
    and `huggingface_hub` names the snapshot directory after the commit the request
    resolved to, so for a SHA request the two agree by construction. What pins the revision
    here is the pair of digests below: content hashing to those values is the pinned
    revision's content. `REVISION_BASIS` states this in every export, so provenance never
    implies a commit lookup that did not run.
    """
    import json

    root = Path(directory)
    if not root.is_dir():
        raise IntegrityError("SNAPSHOT_MISSING", f"snapshot directory not found: {root}")

    forbidden = find_forbidden_weight_files(root)
    if forbidden:
        raise IntegrityError(
            "FORBIDDEN_WEIGHT_FILE",
            f"snapshot contains pickle weight file(s) {forbidden}; the standard path "
            "must load model.safetensors and must never risk a silent .bin fallback",
            {"files": forbidden},
        )

    if has_manifest(root):
        # Fleet snapshot directory (``weights/<MODEL_KEY>/``): the revision is carried by the
        # committed manifest rather than by the directory name. ``verify_snapshot`` re-hashes
        # every manifest entry and asserts the manifest digests equal the pinned constants; the
        # per-file assertions below then run on the same bytes with the caller's expectations.
        manifest = verify_snapshot(root)
        if manifest["revision"] != expected_revision:
            raise IntegrityError(
                "REVISION_MISMATCH",
                f"snapshot manifest names {manifest['revision']!r}, "
                f"expected commit {expected_revision!r}",
                {"resolved": manifest["revision"], "expected": expected_revision},
            )
    elif check_directory_name and root.name != expected_revision:
        raise IntegrityError(
            "REVISION_MISMATCH",
            f"snapshot resolved to {root.name!r}, expected commit {expected_revision!r}",
            {"resolved": root.name, "expected": expected_revision},
        )

    config_path = root / "config.json"
    weights_path = root / PINNED_WEIGHTS_FILENAME
    for path in (config_path, weights_path):
        if not path.is_file():
            raise IntegrityError(
                "SNAPSHOT_INCOMPLETE", f"required file missing from snapshot: {path.name}"
            )

    config_sha = sha256_file(config_path)
    config_bytes = config_path.stat().st_size
    if config_sha != expected_config_sha256:
        raise IntegrityError(
            "CONFIG_DIGEST_MISMATCH",
            f"config.json sha256 {config_sha} != expected {expected_config_sha256}",
            {"actual": config_sha, "expected": expected_config_sha256},
        )

    weights_bytes = weights_path.stat().st_size
    if weights_bytes != expected_weights_bytes:
        raise IntegrityError(
            "WEIGHTS_SIZE_MISMATCH",
            f"{PINNED_WEIGHTS_FILENAME} is {weights_bytes} bytes, "
            f"expected {expected_weights_bytes}",
            {"actual": weights_bytes, "expected": expected_weights_bytes},
        )
    weights_sha = sha256_file(weights_path)
    if weights_sha != expected_weights_sha256:
        raise IntegrityError(
            "WEIGHTS_DIGEST_MISMATCH",
            f"{PINNED_WEIGHTS_FILENAME} sha256 {weights_sha} != expected "
            f"{expected_weights_sha256}",
            {"actual": weights_sha, "expected": expected_weights_sha256},
        )

    config = json.loads(config_path.read_text(encoding="utf-8"))
    return config_sha, config_bytes, weights_sha, weights_bytes, config


def verified_snapshot_from_dir(
    directory: str | os.PathLike[str], model_id: str = PINNED_MODEL_ID
) -> VerifiedSnapshot:
    """Verify a snapshot directory *now* and describe it from what was recomputed.

    Every field of the returned `VerifiedSnapshot` is derived from the files on disk —
    digests from `sha256_file`, the revision from the snapshot directory name, the shape
    constants from the parsed `config.json`. Nothing is copied from a caller's assertion.
    """
    config_sha, config_bytes, weights_sha, weights_bytes, config = verify_snapshot_dir(directory)

    seq_len = int(config["seq_len"])
    patch_len = int(config["patch_len"])
    patch_stride = int(config["patch_stride_len"])
    if (seq_len, patch_len, patch_stride) != (SEQUENCE_LENGTH, PATCH_LENGTH, PATCH_STRIDE):
        raise IntegrityError(
            "CONFIG_SHAPE_MISMATCH",
            f"config.json declares seq_len/patch_len/stride {seq_len}/{patch_len}/"
            f"{patch_stride}, expected {SEQUENCE_LENGTH}/{PATCH_LENGTH}/{PATCH_STRIDE}",
            {"seq_len": seq_len, "patch_len": patch_len, "patch_stride_len": patch_stride},
        )

    root = Path(directory)
    # `verify_snapshot_dir` has already asserted `root.name == PINNED_REVISION` (or, for a
    # fleet snapshot directory, that the digest-verified manifest names it), so the revision
    # is a verified fact rather than a claim.
    revision = _read_manifest(root)["revision"] if has_manifest(root) else root.name
    assert_pinned_source(model_id, revision)
    return VerifiedSnapshot(
        path=root,
        model_id=model_id,
        revision=revision,
        config_sha256=config_sha,
        config_bytes=config_bytes,
        weights_path=root / PINNED_WEIGHTS_FILENAME,
        weights_sha256=weights_sha,
        weights_bytes=weights_bytes,
        seq_len=seq_len,
        patch_len=patch_len,
        patch_stride=patch_stride,
        task_name_in_config=str(config.get("task_name")),
    )


def fetch_verified_snapshot(
    model_id: str = PINNED_MODEL_ID,
    revision: str = PINNED_REVISION,
    cache_dir: str | os.PathLike[str] | None = None,
) -> VerifiedSnapshot:
    """Download (or reuse) and fully verify the pinned snapshot."""
    assert_pinned_source(model_id, revision)
    local = snapshot_download(
        repo_id=model_id,
        revision=revision,
        allow_patterns=ALLOW_PATTERNS,
        cache_dir=cache_dir,
    )
    return verified_snapshot_from_dir(local, model_id)


def _selected_encoder_keys(file_keys: list[str], n: int = 5) -> list[str]:
    encoder = sorted(k for k in file_keys if k.startswith("encoder."))
    if len(encoder) < n:
        return encoder
    step = (len(encoder) - 1) / (n - 1)
    return [encoder[round(i * step)] for i in range(n)]


#: What the tensor comparison can and cannot show. Kept next to the proof it qualifies so
#: the claim and the mechanism cannot drift apart (R-6).
WEIGHT_FILE_IDENTITY_BASIS = (
    "file identity rests on the exclusion controls -- allow_patterns never fetches a "
    "pickle file, and verify_snapshot_dir refuses any snapshot containing one -- NOT on "
    "the tensor comparison below: pytorch_model.bin at this revision is a value-identical "
    "serialization of the same checkpoint and would satisfy a value comparison"
)


def prove_pinned_weights_are_live(
    pipeline: Any, weights_path: str | os.PathLike[str], task: str
) -> dict:
    """Prove the live module holds exactly the tensors in the pinned `model.safetensors`.

    Compares **every** non-`head.*` tensor in the file — and, for the reconstruction task,
    every `head.*` tensor too — against `safetensors.safe_open` entries with `torch.equal`.
    A re-initialized head, a truncated or tampered file, a partial load or the wrong task's
    head all fail here.

    What this does **not** prove is *which file on disk* was read. `pytorch_model.bin` at
    the pinned revision holds the same values, so a `.bin` load would pass this comparison.
    The real defence against a pickle fallback is the pair of exclusion controls named in
    `WEIGHT_FILE_IDENTITY_BASIS`, and they are mutation-tested separately.
    """
    import torch
    from safetensors import safe_open

    state = pipeline.state_dict()
    with safe_open(str(weights_path), framework="pt", device="cpu") as handle:
        file_keys = sorted(handle.keys())
        body_keys = [k for k in file_keys if not k.startswith("head.")]
        if len(_selected_encoder_keys(file_keys)) < 3:
            raise IntegrityError(
                "PROOF_UNAVAILABLE",
                f"{weights_path} exposes {len(_selected_encoder_keys(file_keys))} encoder "
                "tensors; cannot prove the load",
            )
        compared_body: list[str] = []
        for key in body_keys:
            if key not in state:
                raise IntegrityError(
                    "PROOF_KEY_MISSING", f"loaded module has no tensor {key!r}", {"key": key}
                )
            if not torch.equal(state[key].detach().cpu(), handle.get_tensor(key)):
                raise IntegrityError(
                    "PROOF_TENSOR_MISMATCH",
                    f"tensor {key!r} differs from the model.safetensors entry",
                    {"key": key},
                )
            compared_body.append(key)
        checked_encoder = _selected_encoder_keys(file_keys)
        live_not_in_file = sorted(
            k for k in state if not k.startswith("head.") and k not in set(file_keys)
        )

        head_keys = sorted(k for k in file_keys if k.startswith("head."))
        checked_head: list[str] = []
        if task == "reconstruction":
            for key in head_keys:
                if key not in state:
                    raise IntegrityError(
                        "PROOF_HEAD_MISSING",
                        f"reconstruction head tensor {key!r} is absent from the loaded "
                        "module; the pretrained head was not restored",
                        {"key": key},
                    )
                if not torch.equal(state[key].detach().cpu(), handle.get_tensor(key)):
                    raise IntegrityError(
                        "PROOF_TENSOR_MISMATCH",
                        f"head tensor {key!r} differs from the model.safetensors entry",
                        {"key": key},
                    )
                checked_head.append(key)
        else:
            live_head_keys = [k for k in state if k.startswith("head.")]
            if live_head_keys:
                raise IntegrityError(
                    "UNEXPECTED_HEAD_PARAMETERS",
                    f"task {task!r} must expose no head parameters, found {live_head_keys}",
                    {"keys": live_head_keys},
                )

    return {
        "weight_file": Path(weights_path).name,
        "n_tensors_in_file": len(file_keys),
        "n_tensors_compared": len(compared_body) + len(checked_head),
        "encoder_tensors_checked": checked_encoder,
        "head_tensors_checked": checked_head,
        "live_tensors_not_in_file": live_not_in_file,
        "method": (
            "torch.equal against safetensors.safe_open entries for every non-head tensor "
            "in the file (plus every head tensor for the reconstruction task)"
        ),
        "proves": (
            "the live module's values are the pinned checkpoint's values -- no fresh "
            "initialization, no partial or tampered load"
        ),
        "does_not_prove": "which file on disk was read",
        "file_identity_basis": WEIGHT_FILE_IDENTITY_BASIS,
    }


def load_moment(
    task: Task = "embedding",
    device: str = "auto",
    dtype: str = "float32",
    snapshot: VerifiedSnapshot | None = None,
    cache_dir: str | os.PathLike[str] | None = None,
    weights_dir: str | os.PathLike[str] | None = None,
    allow_download: bool = False,
) -> LoadedMoment:
    """Load a task-specific MOMENT instance from the verified pinned snapshot.

    ``weights_dir`` names a fleet snapshot directory holding ``dimer-base-manifest.json``
    (normally ``weights/<MODEL_KEY>/``). When given, nothing goes through
    ``snapshot_download``: manifest entries that are absent are staged with
    :func:`stage_missing_files` (only when ``allow_download=True``), :func:`verify_snapshot`
    re-hashes every entry against the manifest *and* the pinned digests, and the same
    ``MOMENTPipeline.from_pretrained`` call loads from that directory.

    Task instances are explicit because `MOMENTPipeline.init()` replaces the head when
    the task changes (RFC M-8): `reconstruction` keeps the pretrained `PretrainHead`;
    `embedding` swaps in `nn.Identity` and emits a harmless upstream warning that only
    concerns *heads*, never the encoder that produces the embeddings.

    A caller-supplied `snapshot` is **re-verified on disk** before it is used, and the
    identity written into provenance is rebuilt from the recomputed digests. Provenance
    must record what was verified at load time, not what the caller asserted (R-3).
    """
    import torch
    from momentfm import MOMENTPipeline

    if task not in _TASK_TO_UPSTREAM:
        raise ModelSourceError(
            f"v1 exposes only {sorted(_TASK_TO_UPSTREAM)}; refused task {task!r}. "
            "Forecasting and classification heads are NOT pretrained (RFC M-1/M-4)."
        )
    # Same guards as `MomentConfig.__post_init__`, so the public loader is not a way
    # around the validated configuration surface (R-12).
    if device not in SUPPORTED_DEVICES:
        raise ConfigError(f"device must be one of {'|'.join(SUPPORTED_DEVICES)}, got {device!r}")
    if dtype not in SUPPORTED_DTYPES:
        raise ConfigError(_UNSUPPORTED_DTYPE.format(dtype=dtype))

    if weights_dir is not None:
        if snapshot is not None:
            raise ModelSourceError("pass either weights_dir or snapshot, not both")
        stage_missing_files(weights_dir, allow_download=allow_download)
        snapshot = verified_snapshot_from_dir(weights_dir)
    elif snapshot is None:
        snapshot = fetch_verified_snapshot(cache_dir=cache_dir)
    else:
        snapshot = verified_snapshot_from_dir(snapshot.path, snapshot.model_id)

    pipeline = MOMENTPipeline.from_pretrained(
        str(snapshot.path), model_kwargs={"task_name": _TASK_TO_UPSTREAM[task]}
    )
    pipeline.init()
    pipeline.eval()

    proof = prove_pinned_weights_are_live(pipeline, snapshot.weights_path, task)
    if task == "embedding":
        head_type = type(pipeline.head).__name__
        if head_type != "Identity":
            raise IntegrityError(
                "UNEXPECTED_HEAD",
                f"embedding task must use nn.Identity, found {head_type}",
                {"head": head_type},
            )
        proof["head_type"] = head_type
    else:
        proof["head_type"] = type(pipeline.head).__name__

    d_model = int(pipeline.config.d_model)
    resolved_device = device
    if resolved_device == "auto":
        resolved_device = "cuda" if torch.cuda.is_available() else "cpu"
    # Only float32 is reachable (see `config.SUPPORTED_DTYPES`). Kept as a lookup rather
    # than a literal so that widening the surface has to add the entry here too, next to
    # the input-cast requirement the guard above documents.
    torch_dtype = {"float32": torch.float32}[dtype]
    pipeline = pipeline.to(device=resolved_device, dtype=torch_dtype)
    pipeline.eval()

    identity = ModelIdentity(
        name=snapshot.model_id,
        revision=snapshot.revision,
        config_sha256=snapshot.config_sha256,
        weights_sha256=snapshot.weights_sha256,
        weights_bytes=snapshot.weights_bytes,
        license=MODEL_LICENSE,
        license_basis=MODEL_LICENSE_BASIS,
        weight_file_loaded=PINNED_WEIGHTS_FILENAME,
        seq_len=snapshot.seq_len,
        patch_len=snapshot.patch_len,
        patch_stride=snapshot.patch_stride,
        d_model_effective=d_model,
        task=task,
        device=resolved_device,
        dtype=dtype,
    )
    return LoadedMoment(pipeline=pipeline, identity=identity, snapshot=snapshot, proof=proof)

**Module 3/12:** `src/moment_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Labelled-window dataset contract for adapting the encoder: the pinned UCI HAPT sample, validation,
seeded user-level splitting, the long-frame bridge to the canonical windowing path, BYOD loaders and
CSV export.

The default dataset is **real**: 179 ten-second windows of smartphone motion data from the UCI
*Smartphone-Based Recognition of Human Activities and Postural Transitions* dataset (HAPT, CC BY 4.0) —
for each of its 30 volunteers and each of the six basic activities (walking, walking upstairs, walking
downstairs, sitting, standing, laying) the centre 512 samples (10.24 s at 50 Hz) of the first labelled
segment long enough to hold them (one volunteer has no downstairs segment that long, hence 179), over six
channels (total accelerometer x/y/z in g, gyroscope x/y/z in rad/s). The
single 79.6 MB archive is pinned by byte size and SHA-256, fetched from the UCI repository at run time and
refused on any mismatch; the raw text files are read from the archive, never extracted; the repository
redistributes none of it. Windows of one volunteer share a body and a phone, so the sample is split
**by user**, never by window.

A record is ``{id, x, label}``: a float32 ``(n_channels, 512)`` array (or a path to a ``.npy``) and its
activity key. ``user`` (or ``group``) is the split unit when present. ``records_to_long_frame`` turns
records into the ``series_id, timestamp, channel, value`` frame that `validation.validate_long_frame`
and `canonical.to_windows` — the same two calls every inference tutorial makes — validate and window.
"""

# ruff: noqa: E501  -- contract prose and error messages are kept on one line for grep-ability
from __future__ import annotations

import csv
import hashlib
import io
import json
import random
import re
import urllib.request
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

# standalone rewrite (build_notebook.py): `from .config import SEQUENCE_LENGTH` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "UCI HAPT smartphone motion windows (six activities)"
CORPUS_RELEASE = (
    "UCI Machine Learning Repository dataset 341 (Reyes-Ortiz, Anguita, Oneto, Parra 2015), "
    "static archive pinned 2026-09-19"
)
CORPUS_URL = (
    "https://archive.ics.uci.edu/static/public/341/"
    "smartphone+based+recognition+of+human+activities+and+postural+transitions.zip"
)
CORPUS_BYTES = 79_596_192
CORPUS_SHA256 = "4ac4ae064227c07045a99551876b54d204837e995e298b6488559e209b3deb09"
CORPUS_LICENSE = "CC BY 4.0 (UCI Machine Learning Repository licence notice for dataset 341)"
CORPUS_ARCHIVE_NAME = "hapt.zip"
SAMPLE_RATE_HZ = 50
WINDOW_LENGTH = SEQUENCE_LENGTH  # 512 samples = 10.24 s
ACTIVITIES: dict[int, str] = {
    1: "walking",
    2: "walking_upstairs",
    3: "walking_downstairs",
    4: "sitting",
    5: "standing",
    6: "laying",
}
CHANNELS: tuple[str, ...] = ("acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z")
N_USERS = 30
DEFAULT_CACHE_DIR = Path("weights") / "hapt"  # working-directory-relative, like the notebook
SAMPLE_SEED = 42
SAMPLE_SPLIT = {
    "train": 18,
    "validation": 6,
    "test": 6,
}  # users; 6 activities each -> 108 / 36 / 36 windows
MIN_RECORDS = 8
MAX_RECORDS = 1_024  # the canonical path's max_windows
MIN_CLASSES = 2
MAX_CLASSES = 100
MAX_CHANNELS = 32
MAX_LABEL_CHARS = 64
MAX_ABS_VALUE = 1_000.0
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")
_LABEL_RE = re.compile(r"^[A-Za-z0-9 _.,'()/&+-]{1,64}$")
_EPOCH = np.datetime64("2000-01-01T00:00:00", "ns")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> bytes:
    """Return the pinned HAPT archive bytes from the cache or the UCI repository, digest-verified."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    local = cache / CORPUS_ARCHIVE_NAME
    data = local.read_bytes() if local.is_file() else b""
    if len(data) != CORPUS_BYTES or _sha256_bytes(data) != CORPUS_SHA256:
        if fetcher is not None:
            data = fetcher(CORPUS_URL)
        else:
            request = urllib.request.Request(
                CORPUS_URL, headers={"User-Agent": "dimer-moment-tutorial/1.0"}
            )
            with urllib.request.urlopen(request, timeout=600) as response:  # noqa: S310 (pinned https URL)
                data = response.read()
        if len(data) != CORPUS_BYTES or _sha256_bytes(data) != CORPUS_SHA256:
            raise ValueError(
                f"{CORPUS_ARCHIVE_NAME}: fetched {len(data)} bytes with sha256 {_sha256_bytes(data)[:16]}…, "
                f"pinned {CORPUS_BYTES} / {CORPUS_SHA256[:16]}…"
            )
        local.write_bytes(data)
    return data


def _member(archive: zipfile.ZipFile, name: str) -> str:
    matches = [n for n in archive.namelist() if n.endswith(name)]
    if len(matches) != 1:
        raise ValueError(f"archive must hold exactly one {name!r}, found {len(matches)}")
    return matches[0]


def _read_signal(archive: zipfile.ZipFile, member: str) -> np.ndarray:
    text = archive.read(member).decode("utf-8")
    return np.loadtxt(io.StringIO(text), dtype=np.float32)


def read_corpus(payload: bytes) -> list[dict[str, Any]]:
    """Cut the 179 pinned windows out of the verified archive: per user and activity, the centre 512
    samples of the first labelled segment (by experiment, then start) at least 512 samples long."""
    archive = zipfile.ZipFile(io.BytesIO(payload))
    labels = np.loadtxt(
        io.StringIO(archive.read(_member(archive, "RawData/labels.txt")).decode("utf-8")), dtype=int
    )
    if labels.ndim != 2 or labels.shape[1] != 5:
        raise ValueError(
            "labels.txt must have five integer columns (experiment, user, activity, start, end)"
        )
    chosen: dict[tuple[int, int], tuple[int, int, int]] = {}
    for experiment, user, activity, start, end in labels[np.lexsort((labels[:, 3], labels[:, 0]))]:
        key = (int(user), int(activity))
        if activity not in ACTIVITIES or key in chosen or end - start + 1 < WINDOW_LENGTH:
            continue
        chosen[key] = (int(experiment), int(start), int(end))
    signals: dict[tuple[str, int, int], np.ndarray] = {}
    out = []
    for (user, activity), (experiment, start, end) in sorted(chosen.items()):
        centre = (start + end) // 2
        first = centre - WINDOW_LENGTH // 2
        rows = []
        for sensor in ("acc", "gyro"):
            key = (sensor, experiment, user)
            if key not in signals:
                signals[key] = _read_signal(
                    archive,
                    _member(archive, f"RawData/{sensor}_exp{experiment:02d}_user{user:02d}.txt"),
                )
            rows.append(signals[key][first : first + WINDOW_LENGTH].T)
        x = np.ascontiguousarray(np.concatenate(rows, axis=0), dtype=np.float32)
        if x.shape != (len(CHANNELS), WINDOW_LENGTH):
            raise ValueError(f"user {user} activity {activity}: window shape {x.shape}")
        out.append(
            {
                "id": f"u{user:02d}-{ACTIVITIES[activity]}",
                "x": x,
                "label": ACTIVITIES[activity],
                "user": f"user{user:02d}",
                "experiment": experiment,
                "segment": [start, end],
                "first_sample": first,
                "source": f"RawData/{{acc,gyro}}_exp{experiment:02d}_user{user:02d}.txt",
            }
        )
    return out


def records_to_long_frame(records: Sequence[Mapping[str, Any]]) -> pd.DataFrame:
    """The `series_id, timestamp, channel, value` frame of validated records (50 Hz stamps from a fixed
    epoch), one 512-sample series per record, in the shape `validate_long_frame` and `to_windows` take."""
    frames = []
    stamps = _EPOCH + (np.arange(WINDOW_LENGTH) * (1_000_000_000 // SAMPLE_RATE_HZ)).astype(
        "timedelta64[ns]"
    )
    for record in records:
        x = np.asarray(record["x"], dtype=np.float32)
        names = list(record.get("channels") or CHANNELS[: x.shape[0]])
        for c, name in enumerate(names):
            frames.append(
                pd.DataFrame(
                    {
                        "series_id": record["id"],
                        "timestamp": stamps,
                        "channel": name,
                        "value": x[c].astype(np.float64),
                    }
                )
            )
    return pd.concat(frames, ignore_index=True)


def build_sample_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded draw of whole users: `sizes` counts users for train / validation / test."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    rng = random.Random(seed)
    by_user: dict[str, list[dict[str, Any]]] = {}
    for record in records:
        by_user.setdefault(str(record["user"]), []).append(dict(record))
    users = sorted(by_user)
    rng.shuffle(users)
    needed = sum(sizes.values())
    if len(users) < needed:
        raise ValueError(f"only {len(users)} users available, need {needed}")
    out: dict[str, list[dict[str, Any]]] = {name: [] for name in sizes}
    cursor = 0
    for name, per_split in sizes.items():
        for user in users[cursor : cursor + per_split]:
            out[name].extend(by_user[user])
        cursor += per_split
    for name in out:
        rng.shuffle(out[name])
        out[name] = [
            {**r, "id": f"{name}-{i:03d}", "source_id": r["id"]} for i, r in enumerate(out[name])
        ]
    return out


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    fetcher: Any = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned corpus."""
    return build_sample_dataset(
        read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes
    )


def _check_record(record: Any, index: int) -> dict[str, Any]:
    label_name = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label_name} must be a mapping with id/x/label")
    for key in ("id", "x", "label"):
        if key not in record:
            raise ValueError(f"{label_name} is missing {key!r}")
    rid, x, label = record["id"], record["x"], record["label"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label_name}: id must match {_ID_RE.pattern}")
    if isinstance(x, str | Path):
        path = Path(x)
        if not path.is_file():
            raise ValueError(f"{label_name}: x file not found: {path}")
        x = np.load(path, allow_pickle=False)
    if not isinstance(x, np.ndarray) or not np.issubdtype(x.dtype, np.number):
        raise ValueError(f"{label_name}: x must be a numeric numpy array or a path to a .npy file")
    if x.ndim == 1:
        x = x[None, :]
    if x.ndim != 2 or x.shape[1] != WINDOW_LENGTH or not 1 <= x.shape[0] <= MAX_CHANNELS:
        raise ValueError(
            f"{label_name}: x must be (1..{MAX_CHANNELS} channels, {WINDOW_LENGTH}) samples, got shape {x.shape}"
        )
    if not np.isfinite(x).all() or float(np.abs(x).max()) > MAX_ABS_VALUE:
        raise ValueError(f"{label_name}: x must be finite with |value| <= {MAX_ABS_VALUE}")
    if not isinstance(label, str) or not _LABEL_RE.match(label.strip()):
        raise ValueError(
            f"{label_name}: label must be a non-empty string of at most {MAX_LABEL_CHARS} plain characters"
        )
    item = {"id": rid, "x": np.ascontiguousarray(x, dtype=np.float32), "label": label.strip()}
    for key in (
        "source_id",
        "user",
        "group",
        "experiment",
        "segment",
        "first_sample",
        "source",
        "channels",
    ):
        if key in record:
            item[key] = record[key]
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    min_records: int = MIN_RECORDS,
    max_records: int = MAX_RECORDS,
) -> dict[str, Any]:
    """Structural validation of a labelled-window dataset; raises ValueError before any model import."""
    if (
        isinstance(records, Mapping)
        or not isinstance(records, Sequence)
        or isinstance(records, str | bytes)
    ):
        raise ValueError("records must be a list of {id, x, label} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = [_check_record(record, index) for index, record in enumerate(records)]
    ids = [r["id"] for r in checked]
    if len(set(ids)) != len(ids):
        duplicate = next(i for i in ids if ids.count(i) > 1)
        raise ValueError(f"duplicate id {duplicate!r}")
    n_channels = {int(r["x"].shape[0]) for r in checked}
    if len(n_channels) != 1:
        raise ValueError(
            f"every record must have the same channel count, found {sorted(n_channels)}"
        )
    labels = sorted({r["label"] for r in checked})
    if not MIN_CLASSES <= len(labels) <= MAX_CLASSES:
        raise ValueError(
            f"{len(labels)} distinct labels; {MIN_CLASSES}..{MAX_CLASSES} are required"
        )
    counts = {label: sum(1 for r in checked if r["label"] == label) for label in labels}
    return {
        "records": checked,
        "n_records": len(checked),
        "n_channels": n_channels.pop(),
        "window_length": WINDOW_LENGTH,
        "classes": labels,
        "label_counts": counts,
        "users": len({str(r.get("user", r.get("group", r["id"]))) for r in checked}),
        "value_range": {
            "min": float(min(r["x"].min() for r in checked)),
            "max": float(max(r["x"].max() for r in checked)),
        },
        "digest": dataset_digest(checked),
    }


def class_names(records: Sequence[Mapping[str, Any]]) -> list[str]:
    """The sorted label vocabulary of a dataset (the `classes` a head is trained for)."""
    names = sorted({str(r["label"]) for r in records})
    if len(names) < MIN_CLASSES:
        raise ValueError(f"a dataset needs at least {MIN_CLASSES} distinct labels")
    return names


def window_digest(x: np.ndarray) -> str:
    """SHA-256 of the float32 samples (shape + bytes), so a re-saved copy of the same window matches."""
    arr = np.ascontiguousarray(x, dtype=np.float32)
    return _sha256_bytes(f"{arr.shape}:".encode() + arr.tobytes())


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [[r["id"], window_digest(r["x"]), r["label"]] for r in records]
    return _sha256_bytes(
        json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    )


def _split_unit(record: Mapping[str, Any]) -> str:
    return str(record.get("user") or record.get("group") or record.get("source_id") or record["id"])


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no window (by sample digest) and no user appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    units: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = window_digest(record["x"])
            if key in seen and seen[key] != name:
                raise ValueError(f"window {record['id']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
            unit = _split_unit(record)
            if unit in units and units[unit] != name:
                raise ValueError(f"user {unit!r} has windows in both {units[unit]} and {name}")
            units[unit] = name
    return {name: len(records) for name, records in splits.items()}


def user_summary(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Users and label counts per split (an observation of what the split unit was)."""
    return {
        name: {
            "windows": len(records),
            "users": len({_split_unit(r) for r in records}),
            "label_counts": {
                label: sum(1 for r in records if r["label"] == label)
                for label in sorted({r["label"] for r in records})
            },
        }
        for name, records in splits.items()
    }


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD dataset into train/validation/test by split unit (`user` / `group`, else the
    window itself) after de-duplicating windows, so one person's data never straddles splits."""
    if not (
        0.0 <= val_fraction < 1.0
        and 0.0 < test_fraction < 1.0
        and val_fraction + test_fraction < 1.0
    ):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    by_unit: dict[str, list[dict[str, Any]]] = {}
    for record in checked:
        key = window_digest(record["x"])
        if key not in seen:
            seen.add(key)
            by_unit.setdefault(_split_unit(record), []).append(record)
    rng = random.Random(seed)
    units = sorted(by_unit)
    rng.shuffle(units)
    n_test = max(1, round(len(units) * test_fraction))
    n_val = round(len(units) * val_fraction)
    splits: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}
    for name, chosen in (
        ("test", units[:n_test]),
        ("validation", units[n_test : n_test + n_val]),
        ("train", units[n_test + n_val :]),
    ):
        for unit in chosen:
            splits[name].extend(by_unit[unit])
    for part in splits.values():
        rng.shuffle(part)
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(
            f"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required"
        )
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read `{id, x, label}` records from a directory or a zip holding `records.csv` (columns `id`, `file`,
    `label`, optional `group`) beside `.npy` windows of shape (channels, 512); arrays are decoded from the
    archive, never extracted to disk."""
    source = Path(path)
    if source.is_dir():
        table = (source / "records.csv").read_text(encoding="utf-8")
        loader = lambda name: (source / name).read_bytes()  # noqa: E731
    elif source.is_file() and source.suffix.lower() == ".zip":
        archive = zipfile.ZipFile(source)
        members = {Path(n).name: n for n in archive.namelist()}
        if "records.csv" not in members:
            raise ValueError("BYOD zip must contain records.csv")
        table = archive.read(members["records.csv"]).decode("utf-8")
        loader = lambda name: archive.read(members[name])  # noqa: E731
    else:
        raise ValueError(
            "BYOD datasets must be a directory or a .zip holding records.csv and the .npy windows"
        )
    rows = list(csv.DictReader(io.StringIO(table)))
    missing = {"id", "file", "label"} - set(rows[0].keys() if rows else set())
    if missing:
        raise ValueError(f"records.csv is missing columns {sorted(missing)}")
    out = []
    for row in rows:
        item: dict[str, Any] = {
            "id": row["id"],
            "x": np.load(io.BytesIO(loader(row["file"])), allow_pickle=False),
            "label": row["label"],
        }
        if row.get("group"):
            item["group"] = row["group"]
        out.append(item)
    return out


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write the records table of a split (id, file, label, group, provenance) in the shape BYOD expects."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=["id", "file", "label", "group", "experiment", "first_sample", "source"],
        )
        writer.writeheader()
        for record in records:
            writer.writerow(
                {
                    "id": record["id"],
                    "file": f"{record.get('source_id') or record['id']}.npy",
                    "label": record["label"],
                    "group": _split_unit(record),
                    "experiment": record.get("experiment", ""),
                    "first_sample": record.get("first_sample", ""),
                    "source": record.get("source", ""),
                }
            )
    return out

**Module 4/12:** `src/moment_pipeline/validation.py` (carried verbatim; see the note above)

In [ ]:
"""Common validation contract for long-format BYOD input.

Implements rules 1-12 of the RFC "Common validation contract". Every rule raises a
`ValidationError` carrying a stable machine-readable `code`, except rule 7
(frequency irregularity), which the RFC says must be *surfaced* rather than
rejected — it is reported as a flag and only escalated to an error when
`MomentConfig.strict_frequency` is set.

This module depends only on numpy/pandas: the unit suite around it runs with no
network and no torch.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd

# standalone rewrite (build_notebook.py): `from .config import MomentConfig` removed — names are kernel globals defined by the carried modules

REQUIRED_COLUMNS: tuple[str, ...] = ("series_id", "timestamp", "channel", "value")


class ValidationError(Exception):
    """A violated input contract rule.

    Attributes:
        code: stable identifier, e.g. ``DUPLICATE_ROWS``. Safe to branch on.
        message: human-readable explanation.
        details: structured context (offending ids/positions). Never contains raw
            user payload values beyond the minimum needed to locate the problem.
    """

    def __init__(self, code: str, message: str, details: dict[str, Any] | None = None):
        self.code = code
        self.message = message
        self.details: dict[str, Any] = details or {}
        super().__init__(f"[{code}] {message}")


@dataclass(frozen=True)
class SeriesFrequency:
    """Per-series timestamp regularity (RFC rule 7 — surfaced, not silently fixed)."""

    series_id: str
    n_timestamps: int
    regular: bool
    distinct_deltas: tuple[str, ...]


@dataclass(frozen=True)
class ValidationReport:
    """Outcome of a successful validation pass."""

    n_rows: int
    series_ids: tuple[str, ...]
    channels: tuple[str, ...]
    frequencies: tuple[SeriesFrequency, ...]
    fully_missing_series_channels: tuple[tuple[str, str], ...]

    @property
    def irregular_series(self) -> tuple[str, ...]:
        return tuple(f.series_id for f in self.frequencies if not f.regular)

    @property
    def has_irregular_frequency(self) -> bool:
        return bool(self.irregular_series)


def _require_columns(df: pd.DataFrame) -> None:
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValidationError(
            "MISSING_COLUMNS",
            f"long-format input requires columns {list(REQUIRED_COLUMNS)}; missing {missing}",
            {"missing": missing, "present": list(map(str, df.columns))},
        )


def _check_ids(df: pd.DataFrame) -> None:
    for column in ("series_id", "channel"):
        null_positions = np.flatnonzero(df[column].isna().to_numpy())
        if null_positions.size:
            raise ValidationError(
                "NULL_ID",
                f"{column} must be non-null; {null_positions.size} null value(s) found",
                {"column": column, "row_positions": null_positions[:20].tolist()},
            )
        empty = np.flatnonzero(
            df[column].astype("string").str.strip().fillna("").eq("").to_numpy()
        )
        if empty.size:
            raise ValidationError(
                "NULL_ID",
                f"{column} must be non-empty; {empty.size} blank value(s) found",
                {"column": column, "row_positions": empty[:20].tolist()},
            )


def _parse_timestamps(df: pd.DataFrame) -> pd.Series:
    raw = df["timestamp"]
    null_positions = np.flatnonzero(raw.isna().to_numpy())
    if null_positions.size:
        raise ValidationError(
            "NULL_TIMESTAMP",
            f"timestamp must be non-null; {null_positions.size} null value(s) found",
            {"row_positions": null_positions[:20].tolist()},
        )
    if pd.api.types.is_datetime64_any_dtype(raw):
        parsed = raw.astype("datetime64[ns]")
    else:
        parsed = pd.to_datetime(raw, errors="coerce", format="mixed")
    bad = np.flatnonzero(parsed.isna().to_numpy())
    if bad.size:
        raise ValidationError(
            "UNPARSEABLE_TIMESTAMP",
            f"{bad.size} timestamp value(s) could not be parsed",
            {"row_positions": bad[:20].tolist()},
        )
    return parsed


def _coerce_values(df: pd.DataFrame) -> pd.Series:
    """Strict numeric policy: nothing is silently turned into a missing value.

    A value is acceptable if it is already missing (NaN/None -> missing, carried by
    the mask) or parses as a finite float. Anything that would *become* NaN through
    coercion, and any +/-inf, is an error.
    """
    raw = df["value"]
    originally_missing = raw.isna().to_numpy()
    coerced = pd.to_numeric(raw, errors="coerce")
    newly_missing = np.flatnonzero(coerced.isna().to_numpy() & ~originally_missing)
    if newly_missing.size:
        raise ValidationError(
            "NON_NUMERIC_VALUE",
            f"{newly_missing.size} value(s) are not numeric under the strict policy",
            {"row_positions": newly_missing[:20].tolist()},
        )
    numeric = coerced.astype("float64")
    values = numeric.to_numpy(dtype="float64", na_value=np.nan)
    non_finite = np.flatnonzero(~np.isnan(values) & ~np.isfinite(values))
    if non_finite.size:
        raise ValidationError(
            "NON_FINITE_VALUE",
            f"{non_finite.size} value(s) are +/-inf; use a missing value instead",
            {"row_positions": non_finite[:20].tolist()},
        )
    return numeric


def _check_duplicates(frame: pd.DataFrame) -> None:
    duplicated = frame.duplicated(subset=["series_id", "channel", "timestamp"], keep=False)
    if bool(duplicated.any()):
        offenders = (
            frame.loc[duplicated, ["series_id", "channel", "timestamp"]]
            .astype(str)
            .drop_duplicates()
            .head(20)
            .to_dict("records")
        )
        raise ValidationError(
            "DUPLICATE_ROWS",
            f"{int(duplicated.sum())} row(s) duplicate a (series_id, channel, timestamp) key",
            {"examples": offenders},
        )


def _check_limits(frame: pd.DataFrame, config: MomentConfig) -> None:
    limits = config.limits
    n_series = frame["series_id"].nunique()
    n_channels = frame["channel"].nunique()
    if n_series > limits.max_series:
        raise ValidationError(
            "LIMIT_EXCEEDED",
            f"{n_series} series exceeds max_series={limits.max_series}",
            {"limit": "max_series", "actual": int(n_series), "allowed": limits.max_series},
        )
    if n_channels > limits.max_channels:
        raise ValidationError(
            "LIMIT_EXCEEDED",
            f"{n_channels} channels exceeds max_channels={limits.max_channels}",
            {"limit": "max_channels", "actual": int(n_channels), "allowed": limits.max_channels},
        )


def normalize_long_frame(df: pd.DataFrame, config: MomentConfig | None = None) -> pd.DataFrame:
    """Validate and return a normalized copy: string ids, datetime64 stamps, float values."""
    config = config or MomentConfig()
    _require_columns(df)
    if len(df) == 0:
        raise ValidationError("EMPTY_INPUT", "input frame has no rows", {"n_rows": 0})
    if len(df) > config.limits.max_rows:
        raise ValidationError(
            "LIMIT_EXCEEDED",
            f"{len(df)} rows exceeds max_rows={config.limits.max_rows}",
            {"limit": "max_rows", "actual": len(df), "allowed": config.limits.max_rows},
        )
    df = df.reset_index(drop=True)
    _check_ids(df)
    frame = pd.DataFrame(
        {
            "series_id": df["series_id"].astype("string").astype("object").map(str),
            "channel": df["channel"].astype("string").astype("object").map(str),
            "timestamp": _parse_timestamps(df),
            "value": _coerce_values(df),
        }
    )
    _check_duplicates(frame)
    _check_limits(frame, config)
    return frame


def validate_long_frame(
    df: pd.DataFrame, config: MomentConfig | None = None
) -> tuple[ValidationReport, pd.DataFrame]:
    """Apply the full common validation contract.

    Returns the report and the normalized frame so callers do not re-parse. Raises
    `ValidationError` on the first violated rule.
    """
    config = config or MomentConfig()
    frame = normalize_long_frame(df, config)

    channels = tuple(sorted(frame["channel"].unique().tolist()))
    series_ids = tuple(sorted(frame["series_id"].unique().tolist()))

    observed_per_channel = frame.loc[frame["value"].notna(), "channel"].unique().tolist()
    empty_channels = [c for c in channels if c not in set(observed_per_channel)]
    if empty_channels:
        raise ValidationError(
            "EMPTY_CHANNEL",
            f"channel(s) {empty_channels} contain no observed values anywhere in the input",
            {"channels": empty_channels},
        )

    frequencies: list[SeriesFrequency] = []
    for series_id in series_ids:
        selected = frame.loc[frame["series_id"] == series_id, "timestamp"]
        stamps = np.sort(selected.unique().astype("datetime64[ns]"))
        deltas = np.diff(stamps)
        distinct = np.unique(deltas)
        frequencies.append(
            SeriesFrequency(
                series_id=series_id,
                n_timestamps=int(stamps.size),
                regular=bool(distinct.size <= 1),
                distinct_deltas=tuple(str(d) for d in distinct[:10]),
            )
        )

    fully_missing: list[tuple[str, str]] = []
    observed = frame.loc[frame["value"].notna(), ["series_id", "channel"]]
    observed_pairs = set(map(tuple, observed.drop_duplicates().to_numpy().tolist()))
    for series_id in series_ids:
        for channel in channels:
            if (series_id, channel) not in observed_pairs:
                fully_missing.append((series_id, channel))

    report = ValidationReport(
        n_rows=len(frame),
        series_ids=series_ids,
        channels=channels,
        frequencies=tuple(frequencies),
        fully_missing_series_channels=tuple(fully_missing),
    )

    # RFC rule 9, third case. `EMPTY_CHANNEL` catches a channel empty across the whole
    # input and `ALL_MISSING_WINDOW` catches a window empty in every channel, but a single
    # (series, channel) pair that is entirely missing used to be reported and then passed
    # straight through. Downstream its `point_mask` row is all zero, so the collapsed
    # `model_point_mask` and every `patch_mask` entry for that window are zero, upstream
    # RevIN takes `nanmean` over an all-NaN row (momentfm/models/layers/revin.py:58-65) and
    # `reconstruct` finally raises an M-2 error blaming the pre-fill -- which is not the
    # cause. `embed` did not even complain: it returned a finite embedding of a fabricated
    # all-zero channel. Reject it here, where the cause is known (R-4).
    if fully_missing:
        raise ValidationError(
            "FULLY_MISSING_SERIES_CHANNEL",
            "these (series_id, channel) pairs have no observed value at all: "
            f"{[list(pair) for pair in fully_missing]}. MOMENT would receive a fabricated "
            "all-prefill channel with an all-zero mask; drop the channel, drop the series, "
            "or impute before the pipeline",
            {"pairs": [list(pair) for pair in fully_missing]},
        )

    if config.strict_frequency and report.has_irregular_frequency:
        raise ValidationError(
            "IRREGULAR_FREQUENCY",
            "strict_frequency=True and these series have non-uniform timestamp spacing: "
            f"{list(report.irregular_series)}",
            {"series_ids": list(report.irregular_series)},
        )
    return report, frame

**Module 5/12:** `src/moment_pipeline/canonical.py` (carried verbatim; see the note above)

In [ ]:
"""Long-format -> canonical MOMENT tensor conversion.

Documented policy for the pinned `AutonLab/MOMENT-1-base` checkpoint (seq_len 512,
patch_len 8, stride 8):

* **Channel ordering** — the sorted unique channel names of the *whole* input. Every
  window therefore shares one channel axis, and the order does not depend on row order.
* **Windowing (Phase 1)** — exactly ONE window per `series_id`: the last 512 distinct
  timestamps of that series, right-aligned.
* **Truncation** — a series with more than 512 distinct timestamps keeps only the last
  512; `WindowSet.truncated` flags it, so the loss is disclosed before inference.
* **Padding** — a series with fewer than 512 timestamps is LEFT-padded. Padded positions
  get `input_mask = 0` (MOMENT's RevIN normalizer ignores them) and `point_mask = 0`.
* **Missing values** — mandatory finite pre-fill. Missing payloads are replaced with
  `MomentConfig.prefill_value` (default 0.0) *before* the tensor exists; missingness
  survives only in `point_mask`. Raw NaN never reaches MOMENT (RFC M-2, rules 11-12).
* **Frequency** — irregular spacing is surfaced in the validation report, never
  interpolated (RFC rule 7).
* **Normalization** — delegated to MOMENT's internal RevIN, which is driven by
  `input_mask`. This layer performs no scaling of its own.

Mask vocabulary (1 = usable, 0 = not), all `float32`:

| array              | shape             | meaning                                        |
|--------------------|-------------------|------------------------------------------------|
| `input_mask`       | (b, 512)          | position belongs to the series (not padding)   |
| `point_mask`       | (b, c, 512)       | value observed rather than pre-filled          |
| `model_point_mask` | (b, 512)          | `point_mask` collapsed over channels (min)     |
| `patch_mask`       | (b, 64)           | patch fully observed (upstream patch view)     |
"""

from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np
import pandas as pd

# standalone rewrite (build_notebook.py): `from .config import MomentConfig` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .validation import ValidationError, ValidationReport, validate_long_frame` removed — names are kernel globals defined by the carried modules


def to_patch_view(point_mask: np.ndarray, patch_length: int = 8) -> np.ndarray:
    """Reproduce `momentfm.utils.masking.Masking.convert_seq_to_patch_view`.

    A patch counts as observed only when all `patch_length` of its points are observed
    (upstream: ``(mask.unfold(...).sum(dim=-1) == patch_len)``). This is the quantization
    that makes one missing point cost a whole patch (RFC M-3).
    """
    if point_mask.ndim != 2:
        raise ValueError(f"expected a (batch, seq_len) mask, got shape {point_mask.shape}")
    batch, seq_len = point_mask.shape
    if seq_len % patch_length:
        raise ValueError(f"seq_len {seq_len} is not a multiple of patch_length {patch_length}")
    folded = point_mask.reshape(batch, seq_len // patch_length, patch_length)
    return (folded.sum(axis=-1) == patch_length).astype(np.float32)


def expand_patch_view(patch_mask: np.ndarray, patch_length: int = 8) -> np.ndarray:
    """Expand a (batch, n_patches) patch mask back to (batch, seq_len) point granularity."""
    return np.repeat(patch_mask, patch_length, axis=1).astype(np.float32)


# --- the single definition of each reported fraction (R-5) --------------------------
#
# Two call sites used to compute a quantity called `masked_point_fraction` with different
# denominators -- `WindowSet` counted per-channel cells, `ReconstructionResult` broadcast a
# channel-collapsed mask back over the channel axis, so one missing point in one channel of
# a C-channel window counted C times. One missing point at index 13 of `c1` in a 2-channel
# window gave 1/1024 from one and 2/1024 from the other. These three functions are now the
# only definitions; every reported fraction goes through them, and the docstrings state the
# denominator each one uses.


def missing_point_fraction(point_mask: np.ndarray, input_mask: np.ndarray) -> float:
    """Fraction of **non-padded (window, channel, position) cells** that were missing.

    Denominator: `input_mask.sum() * n_channels`. This is source missingness -- what the
    caller's data did not contain -- and it is independent of any extra mask the caller
    later applies for reconstruction.
    """
    valid = np.broadcast_to(input_mask[:, None, :], point_mask.shape)
    denom = float(valid.sum())
    if denom == 0.0:
        return 0.0
    return float(((valid == 1) & (point_mask == 0)).sum()) / denom


def hidden_position_fraction(visible_mask: np.ndarray, input_mask: np.ndarray) -> float:
    """Fraction of **non-padded (window, position) pairs** hidden from the model.

    Denominator: `input_mask.sum()`. There is no channel axis because MOMENT's `mask`
    argument has none: `visible_mask` is already collapsed over channels. This counts what
    the model was told not to look at -- source missingness *and* any mask the caller
    supplied -- so it is a different quantity from `missing_point_fraction` and carries a
    different name.
    """
    denom = float(input_mask.sum())
    if denom == 0.0:
        return 0.0
    return float(((input_mask == 1) & (visible_mask == 0)).sum()) / denom


def masked_patch_fraction_of(
    patch_mask: np.ndarray, input_mask: np.ndarray, patch_length: int = 8
) -> float:
    """Fraction of **non-padded patches** the model will treat as unobserved.

    Denominator: the patch view of `input_mask`.
    """
    valid_patches = to_patch_view(input_mask, patch_length)
    denom = float(valid_patches.sum())
    if denom == 0.0:
        return 0.0
    return float(((valid_patches == 1) & (patch_mask == 0)).sum()) / denom


@dataclass(frozen=True)
class WindowSet:
    """The canonical DIMER-facing representation handed to MOMENT."""

    x_enc: np.ndarray
    input_mask: np.ndarray
    point_mask: np.ndarray
    series_ids: tuple[str, ...]
    window_ids: tuple[str, ...]
    channels: tuple[str, ...]
    truncated: tuple[bool, ...]
    padded: tuple[bool, ...]
    n_source_timestamps: tuple[int, ...]
    timestamps: tuple[np.ndarray, ...] = field(repr=False, default=())
    prefill_value: float = 0.0
    patch_length: int = 8
    validation: ValidationReport | None = field(repr=False, default=None)

    @property
    def n_windows(self) -> int:
        return int(self.x_enc.shape[0])

    @property
    def n_channels(self) -> int:
        return int(self.x_enc.shape[1])

    @property
    def sequence_length(self) -> int:
        return int(self.x_enc.shape[2])

    @property
    def model_point_mask(self) -> np.ndarray:
        """`point_mask` collapsed over channels: a position is observed only if it is
        observed in every channel. MOMENT's `mask` argument has no channel axis, so the
        collapse is conservative and explicit rather than silently per-channel."""
        return self.point_mask.min(axis=1).astype(np.float32)

    @property
    def patch_mask(self) -> np.ndarray:
        return to_patch_view(self.model_point_mask, self.patch_length)

    def patch_quantized_mask(self) -> np.ndarray:
        """The (batch, seq_len) mask actually passed to `pipeline.reconstruct`.

        It is the patch view expanded back to points, so what the caller sees is exactly
        what the model uses — no hidden widening inside momentfm.
        """
        return expand_patch_view(self.patch_mask, self.patch_length)

    @property
    def valid_point_count(self) -> int:
        return int(self.input_mask.sum() * self.n_channels)

    @property
    def masked_point_count(self) -> int:
        """Non-padded (window, channel, position) cells that were missing in the source."""
        valid = np.broadcast_to(self.input_mask[:, None, :], self.point_mask.shape)
        return int(((valid == 1) & (self.point_mask == 0)).sum())

    @property
    def masked_point_fraction(self) -> float:
        """See `missing_point_fraction` -- denominator is non-padded cells x channels."""
        return missing_point_fraction(self.point_mask, self.input_mask)

    @property
    def masked_patch_fraction(self) -> float:
        """See `masked_patch_fraction_of` -- denominator is non-padded patches."""
        return masked_patch_fraction_of(self.patch_mask, self.input_mask, self.patch_length)

    @property
    def padded_fraction(self) -> float:
        return float((self.input_mask == 0).sum()) / float(self.input_mask.size)


def to_windows(
    df: pd.DataFrame,
    config: MomentConfig | None = None,
    report: ValidationReport | None = None,
    frame: pd.DataFrame | None = None,
) -> WindowSet:
    """Validate (unless a report+frame is supplied) and convert to the canonical tensors."""
    config = config or MomentConfig()
    if report is None or frame is None:
        report, frame = validate_long_frame(df, config)

    channels = list(report.channels)
    series_ids = list(report.series_ids)
    seq_len = config.sequence_length

    if len(series_ids) > config.limits.max_windows:
        raise ValidationError(
            "LIMIT_EXCEEDED",
            f"{len(series_ids)} windows exceeds max_windows={config.limits.max_windows}",
            {
                "limit": "max_windows",
                "actual": len(series_ids),
                "allowed": config.limits.max_windows,
            },
        )

    x_rows: list[np.ndarray] = []
    input_rows: list[np.ndarray] = []
    point_rows: list[np.ndarray] = []
    stamp_rows: list[np.ndarray] = []
    truncated: list[bool] = []
    padded: list[bool] = []
    n_source: list[int] = []

    grouped = {sid: sub for sid, sub in frame.groupby("series_id", sort=True)}
    for series_id in series_ids:
        subset = grouped[series_id]
        stamps = np.sort(subset["timestamp"].unique().astype("datetime64[ns]"))
        n_source.append(int(stamps.size))
        truncated.append(bool(stamps.size > seq_len))
        padded.append(bool(stamps.size < seq_len))
        kept = stamps[-seq_len:]

        wide = (
            subset.set_index(["channel", "timestamp"])["value"]
            .unstack("timestamp")
            .reindex(index=channels, columns=pd.DatetimeIndex(kept))
        )
        grid = wide.to_numpy(dtype="float64", na_value=np.nan)
        observed = np.isfinite(grid)
        if not observed.any():
            raise ValidationError(
                "ALL_MISSING_WINDOW",
                f"series {series_id!r} has no observed value in its 512-point window",
                {"series_id": series_id, "n_timestamps": int(stamps.size)},
            )

        filled = np.where(observed, grid, config.prefill_value).astype(np.float32)
        pad = seq_len - kept.size
        if pad > 0:
            filled = np.concatenate(
                [np.full((len(channels), pad), config.prefill_value, np.float32), filled], axis=1
            )
            observed = np.concatenate(
                [np.zeros((len(channels), pad), bool), observed], axis=1
            )
            window_mask = np.concatenate(
                [np.zeros(pad, np.float32), np.ones(kept.size, np.float32)]
            )
            window_stamps = np.concatenate(
                [np.full(pad, np.datetime64("NaT", "ns")), kept]
            )
        else:
            window_mask = np.ones(seq_len, np.float32)
            window_stamps = kept

        x_rows.append(filled)
        input_rows.append(window_mask)
        point_rows.append(observed.astype(np.float32))
        stamp_rows.append(window_stamps)

    x_enc = np.stack(x_rows).astype(np.float32)
    if not np.isfinite(x_enc).all():  # pragma: no cover - defensive, pre-fill guarantees this
        raise ValidationError(
            "NON_FINITE_TENSOR",
            "canonical x_enc contains non-finite values after pre-fill",
            {},
        )

    return WindowSet(
        x_enc=x_enc,
        input_mask=np.stack(input_rows).astype(np.float32),
        point_mask=np.stack(point_rows).astype(np.float32),
        series_ids=tuple(series_ids),
        window_ids=tuple(f"{sid}::w0" for sid in series_ids),
        channels=tuple(channels),
        truncated=tuple(truncated),
        padded=tuple(padded),
        n_source_timestamps=tuple(n_source),
        timestamps=tuple(stamp_rows),
        prefill_value=float(config.prefill_value),
        patch_length=config.patch_length,
        validation=report,
    )

**Module 6/12:** `src/moment_pipeline/csvio.py` (carried verbatim; see the note above)

In [ ]:
"""Strict CSV ingestion for long-format DIMER notebook input.

The notebook specification requires duplicate or otherwise ambiguous CSV headers to
be rejected before pandas can silently mangle them. This module owns that raw-byte
boundary; callers should still pass the returned frame through ``validate_long_frame``
for the production data contract.
"""

from __future__ import annotations

import csv
import io

import pandas as pd

# standalone rewrite (build_notebook.py): `from .validation import REQUIRED_COLUMNS, ValidationError` removed — names are kernel globals defined by the carried modules


def read_long_csv_bytes(payload: bytes) -> pd.DataFrame:
    """Parse UTF-8 CSV bytes only after validating the raw header.

    Header comparison is case-insensitive and ignores surrounding whitespace for
    ambiguity detection, while the actual required column names remain exact. This
    prevents inputs such as ``value,value`` or ``value, Value `` from being silently
    renamed by pandas before the repository validator can inspect the schema.
    """
    try:
        text = payload.decode("utf-8-sig")
    except UnicodeDecodeError as exc:
        raise ValidationError(
            "INVALID_CSV_ENCODING",
            "CSV input must be UTF-8 (an optional UTF-8 BOM is accepted)",
        ) from exc

    reader = csv.reader(io.StringIO(text))
    header = next(reader, None)
    if header is None:
        raise ValidationError("EMPTY_CSV", "CSV input has no header row")

    canonical = [column.strip().casefold() for column in header]
    blank_positions = [index for index, column in enumerate(canonical) if not column]
    if blank_positions:
        raise ValidationError(
            "AMBIGUOUS_COLUMNS",
            "CSV header contains blank column name(s)",
            {"column_positions": blank_positions},
        )

    duplicates = sorted(
        {canonical[index] for index, column in enumerate(canonical) if canonical.count(column) > 1}
    )
    if duplicates:
        raise ValidationError(
            "DUPLICATE_COLUMNS",
            "CSV header contains duplicate or ambiguous column names before dataframe parsing: "
            f"{duplicates}",
            {"columns": duplicates},
        )

    present = set(header)
    missing = [column for column in REQUIRED_COLUMNS if column not in present]
    if missing:
        raise ValidationError(
            "MISSING_COLUMNS",
            f"long-format input requires columns {list(REQUIRED_COLUMNS)}; missing {missing}",
            {"missing": missing, "present": header},
        )

    return pd.read_csv(io.StringIO(text))

**Module 7/12:** `src/moment_pipeline/embedding.py` (carried verbatim; see the note above)

In [ ]:
"""Task 1 — pretrained-encoder embeddings.

Phase 1 exposes the minimum defensible contract: one pooled vector per window from the
*pretrained encoder*, with the reduction recorded in provenance.

Reduction semantics (RFC M-5), read off the installed
`momentfm/models/moment.py::MOMENT.embed`:

* `reduction="mean"` first averages over the **channel** axis
  (`enc_out.mean(dim=1)`), then takes the `input_mask`-weighted mean over patches.
  A multichannel window therefore yields ONE vector in which channels are averaged —
  it is not a per-channel representation and must not be described as one.
* No task head is involved: the embedding loader uses `nn.Identity`, so nothing about
  this path is freshly initialized. The upstream warning "Only reconstruction head is
  pre-trained…" fires for any non-reconstruction task and refers to *heads only*.

**Known limitation — embeddings are not missingness-aware.** Upstream
`MOMENT.embed(self, *, x_enc, input_mask=None, reduction='mean', **kwargs)` takes no
per-point observedness mask; `input_mask` is the *padding* mask, and momentfm feeds it to
both RevIN (`moment.py:254`) and the patch embedding (`moment.py:262`). The finite pre-fill
is therefore consumed as observed data: a window that is half missing produces a vector
byte-identical to the same window with literal zeros in those positions, and both differ
from the clean window. `point_mask` cannot reach the model on this path -- there is no
parameter to carry it. This module therefore records the missingness fractions on the
result and in provenance, which are the only signal an export carries about it. See
`MODEL_CARD.md` and `tests/test_integration_model.py::
test_embeddings_are_missingness_blind_known_limitation`.
"""

from __future__ import annotations

import time
from dataclasses import dataclass, field

import numpy as np
import pandas as pd

# standalone rewrite (build_notebook.py): `from .canonical import WindowSet` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .model import LoadedMoment` removed — names are kernel globals defined by the carried modules

REDUCTION = "mean"

#: Stated on every embedding result and in every embedding provenance block. The point is
#: that the recorded fractions are the *only* place missingness survives on this path.
MISSINGNESS_POLICY = (
    "NOT missingness-aware: upstream MOMENT.embed accepts no per-point observedness mask "
    "(input_mask is the padding mask), so pre-filled positions are seen by the encoder as "
    "observed values and enter RevIN and the patch embedding as data. An embedding of a "
    "window containing missing data equals the embedding of the same window with the "
    "prefill_value written into those positions. masked_point_fraction / "
    "masked_patch_fraction are the only record that any of it was fabricated"
)
CHANNEL_POLICY = (
    "channels are averaged inside momentfm before patch pooling (reduction='mean'); "
    "the result is one vector per window, not one per channel"
)


class TaskMismatchError(RuntimeError):
    """The loaded pipeline was built for a different task than the call requires."""


@dataclass(frozen=True)
class EmbeddingResult:
    embeddings: np.ndarray
    series_ids: tuple[str, ...]
    window_ids: tuple[str, ...]
    d_model: int
    latency_seconds: float
    reduction: str = REDUCTION
    channel_policy: str = CHANNEL_POLICY
    n_channels: int = 1
    truncated: tuple[bool, ...] = field(default=())
    padded: tuple[bool, ...] = field(default=())
    #: Source missingness of the windows these vectors were computed from. Same definition
    #: as `WindowSet.masked_point_fraction` / `.masked_patch_fraction` (R-5): the point
    #: fraction's denominator is non-padded cells x channels, the patch fraction's is
    #: non-padded patches. Nothing about them reached the model -- see MISSINGNESS_POLICY.
    masked_point_fraction: float = 0.0
    masked_point_count: int = 0
    masked_patch_fraction: float = 0.0
    missingness_policy: str = MISSINGNESS_POLICY
    missingness_visible_to_model: bool = False

    def to_frame(self) -> pd.DataFrame:
        """`series_id, window_id, embedding_0 … embedding_n` (RFC Task 1 output)."""
        columns = {f"embedding_{i}": self.embeddings[:, i] for i in range(self.embeddings.shape[1])}
        return pd.DataFrame(
            {"series_id": list(self.series_ids), "window_id": list(self.window_ids), **columns}
        )


def embed(
    windows: WindowSet,
    model: LoadedMoment,
    batch_size: int = 8,
    warmup: bool = True,
) -> EmbeddingResult:
    """Pooled embeddings for every window. Deterministic in eval mode.

    Args:
        warmup: run one discarded forward pass before the timed one so
            `latency_seconds` excludes lazy-allocation cost. Defaults to True for an
            honest latency number; pass False in production, where it otherwise
            doubles the cost of every call (R-9).
    """
    import torch

    if model.identity.task != "embedding":
        raise TaskMismatchError(
            f"embed() needs a pipeline loaded with task='embedding', got "
            f"{model.identity.task!r}; MOMENT replaces its head per task, so reuse of a "
            "reconstruction instance would silently change the computation"
        )
    if windows.sequence_length != model.identity.seq_len:
        raise ValueError(
            f"windows are {windows.sequence_length} long, model expects "
            f"{model.identity.seq_len}"
        )

    device = model.identity.device
    x_all = torch.from_numpy(windows.x_enc)
    mask_all = torch.from_numpy(windows.input_mask)

    def _run() -> np.ndarray:
        chunks: list[np.ndarray] = []
        with torch.no_grad():
            for start in range(0, windows.n_windows, batch_size):
                stop = start + batch_size
                out = model.pipeline.embed(
                    x_enc=x_all[start:stop].to(device),
                    input_mask=mask_all[start:stop].to(device),
                    reduction=REDUCTION,
                )
                chunks.append(out.embeddings.detach().float().cpu().numpy())
        return np.concatenate(chunks, axis=0)

    if warmup:
        _run()  # discarded; keeps latency_seconds free of lazy-allocation cost
    started = time.perf_counter()
    embeddings = _run()
    latency = time.perf_counter() - started

    if not np.isfinite(embeddings).all():
        raise ValueError("embeddings contain non-finite values; the pre-fill contract failed")

    return EmbeddingResult(
        embeddings=embeddings,
        series_ids=windows.series_ids,
        window_ids=windows.window_ids,
        d_model=int(embeddings.shape[1]),
        latency_seconds=latency,
        n_channels=windows.n_channels,
        truncated=windows.truncated,
        padded=windows.padded,
        masked_point_fraction=windows.masked_point_fraction,
        masked_point_count=windows.masked_point_count,
        masked_patch_fraction=windows.masked_patch_fraction,
    )

**Module 8/12:** `src/moment_pipeline/provenance.py` (carried verbatim; see the note above)

In [ ]:
"""Structured provenance attached to every v1 export.

Shape follows the RFC "Output/provenance contract": a `model`, `runtime` and `inference`
block. Two things the RFC template leaves as placeholders are filled in concretely here
because they are the supply-chain claims that actually matter:

* `model.weight_file_loaded` — which weight file the runtime proof confirmed, not which
  one was requested;
* `runtime.momentfm_source` — momentfm is installed from an exact upstream commit, so the
  version string `0.1.5` alone is recorded together with the commit that produced it;
* `model.revision_basis` — how the revision was established. This path pins it by content
  digest, not by an independent Hub commit lookup, and an export that printed `revision`
  alone would imply the stronger check.
"""

from __future__ import annotations

import platform
import sys
import time
from collections.abc import Callable
from importlib.metadata import PackageNotFoundError, version
from typing import Any

# standalone rewrite (build_notebook.py): `from .canonical import WindowSet` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .model import (` removed — names are kernel globals defined by the carried modules


def _version(package: str) -> str | None:
    try:
        return version(package)
    except PackageNotFoundError:  # pragma: no cover - all of these are hard dependencies
        return None


def measure_latency[T](fn: Callable[[], T], warmup: int = 1) -> tuple[T, float]:
    """Run `fn` after `warmup` discarded calls and return (result, seconds)."""
    for _ in range(warmup):
        fn()
    started = time.perf_counter()
    result = fn()
    return result, time.perf_counter() - started


def model_block(identity: ModelIdentity) -> dict[str, Any]:
    return {
        "name": identity.name,
        "revision": identity.revision,
        # A bare revision string reads as a verified commit. On this path it is a content
        # claim, so the basis travels with it rather than being left to the reader.
        "revision_basis": identity.revision_basis,
        "config_sha256": identity.config_sha256,
        "weights_sha256": identity.weights_sha256,
        "weights_bytes": identity.weights_bytes,
        "weight_file_loaded": identity.weight_file_loaded,
        "license": identity.license,
        "license_basis": identity.license_basis,
        "task": identity.task,
        "seq_len": identity.seq_len,
        "patch_len": identity.patch_len,
        "patch_stride": identity.patch_stride,
        "d_model_effective": identity.d_model_effective,
    }


def runtime_block(identity: ModelIdentity) -> dict[str, Any]:
    return {
        "python": platform.python_version(),
        "python_implementation": platform.python_implementation(),
        "platform": sys.platform,
        "momentfm": _version("momentfm"),
        "momentfm_source": {"url": MOMENTFM_SOURCE_URL, "commit": MOMENTFM_SOURCE_COMMIT},
        "huggingface_hub": _version("huggingface-hub"),
        "transformers": _version("transformers"),
        "torch": _version("torch"),
        "numpy": _version("numpy"),
        "device": identity.device,
        "dtype": identity.dtype,
    }


def _inference_common(identity: ModelIdentity, windows: WindowSet) -> dict[str, Any]:
    return {
        "task": identity.task,
        "sequence_length": windows.sequence_length,
        "patch_length": windows.patch_length,
        "n_series": len(set(windows.series_ids)),
        "n_channels": windows.n_channels,
        "n_windows": windows.n_windows,
        "n_truncated_windows": int(sum(windows.truncated)),
        "n_padded_windows": int(sum(windows.padded)),
    }


def build_provenance(model: LoadedMoment, windows: WindowSet, result: Any) -> dict[str, Any]:
    """Assemble the export metadata for one inference call."""
    pass  # standalone rewrite (build_notebook.py): `from .anomaly import AnomalyResult` removed — names are kernel globals defined by the carried modules
    pass  # standalone rewrite (build_notebook.py): `from .embedding import EmbeddingResult` removed — names are kernel globals defined by the carried modules
    pass  # standalone rewrite (build_notebook.py): `from .imputation import ReconstructionResult` removed — names are kernel globals defined by the carried modules

    inference = _inference_common(model.identity, windows)
    inference["latency_seconds"] = round(float(getattr(result, "latency_seconds", 0.0)), 6)

    if isinstance(result, EmbeddingResult):
        inference["reduction"] = result.reduction
        inference["channel_policy"] = result.channel_policy
        inference["embedding_dim"] = result.d_model
        # An exported embedding must be self-describing about the data it was computed
        # from: nothing downstream can tell a fabricated stretch from a real one, because
        # the encoder could not either (R-2).
        inference["masked_point_fraction"] = result.masked_point_fraction
        inference["masked_point_count"] = result.masked_point_count
        inference["masked_patch_fraction"] = result.masked_patch_fraction
        inference["missingness_visible_to_model"] = result.missingness_visible_to_model
        inference["missingness_policy"] = result.missingness_policy
    elif isinstance(result, ReconstructionResult):
        inference["masked_point_fraction"] = result.masked_point_fraction
        inference["masked_point_count"] = result.masked_point_count
        inference["model_masked_point_fraction"] = result.model_masked_point_fraction
        inference["model_masked_point_count"] = result.model_masked_point_count
        inference["masked_patch_fraction"] = result.masked_patch_fraction
        inference["masked_patch_count"] = result.masked_patch_count
        inference["masked_point_fraction_basis"] = (
            "masked_point_fraction counts non-padded (window, channel, position) cells "
            "missing in the source, exactly as WindowSet.masked_point_fraction does; "
            "model_masked_point_fraction counts non-padded (window, position) pairs "
            "hidden from the model, source missingness collapsed over channels plus any "
            "caller-supplied mask"
        )
        inference["mask_policy"] = (
            "explicit patch-quantized mask; mask=None is never passed to "
            "MOMENT.reconstruct"
        )
    elif isinstance(result, AnomalyResult):
        # An anomaly export is only readable next to the loss, the aggregation and the
        # domain the score is defined on. All three travel with it; none has a default a
        # reader could assume, and there is no threshold field because v1 applies none.
        inference["anomaly_loss"] = result.loss
        inference["channel_aggregation"] = result.channel_aggregation
        inference["threshold_policy"] = result.threshold_policy
        inference["score_policy"] = result.score_policy
        inference["scored_domain_policy"] = result.scored_domain_policy
        inference["scored_point_count"] = result.scored_point_count
        inference["scored_point_fraction"] = result.scored_point_fraction
        inference["unscored_prefilled_count"] = result.unscored_prefilled_count
        inference["unscored_hidden_by_patch_count"] = result.unscored_hidden_by_patch_count
        inference["masked_point_fraction"] = result.masked_point_fraction
        inference["masked_patch_fraction"] = result.masked_patch_fraction
    else:  # pragma: no cover - defensive
        raise TypeError(f"unsupported result type {type(result).__name__}")

    return {
        "model": model_block(model.identity),
        "runtime": runtime_block(model.identity),
        "inference": inference,
        "load_proof": {
            key: value for key, value in model.proof.items() if key != "encoder_tensors_checked"
        }
        | {"encoder_tensors_checked": list(model.proof.get("encoder_tensors_checked", []))},
    }

**Module 9/12:** `src/moment_pipeline/adaptation.py` (carried verbatim; see the note above)

In [ ]:
"""Bounded supervised adaptation of the MOMENT encoder to a labelled-window classification task — the
first task-head contract of this repository — under an explicit frozen-vs-unfrozen policy.

MOMENT-1-base ships no classification head (its upstream `classification` task head is *not* pretrained,
RFC M-1/M-4), so the head trained here is the caller's own, on top of the pooled 768-d window embeddings
of the `embedding` task instance. The **frozen policy** trains only that linear head on frozen embeddings
(a linear probe); the **unfrozen policy** continues by training the last `trainable_blocks` T5 encoder
blocks with the head end to end; the epoch with the lowest validation log-loss — which may be the probe
itself — is kept and its tensors restored. Metrics are accuracy and macro-F1 with per-class recall and a
confusion matrix, framed by a majority floor and a cosine k-NN vote over the frozen embeddings.

Everything goes through the repository's own path: records → `records_to_long_frame` → `validate_long_frame`
→ `to_windows` → `embed`; nothing here calls momentfm outside `LoadedMoment.pipeline.embed`.
"""

# ruff: noqa: E501  -- contract prose and error messages are kept on one line for grep-ability
from __future__ import annotations

import hashlib
import json
import math
import random
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np

# standalone rewrite (build_notebook.py): `from .canonical import WindowSet, to_windows` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .config import MomentConfig` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .embedding import embed` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .model import PINNED_WEIGHTS_FILENAME, LoadedMoment` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .samples import MAX_RECORDS, records_to_long_frame, validate_dataset` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .validation import validate_long_frame` removed — names are kernel globals defined by the carried modules

ENCODER_BLOCKS = 12
BLOCK_PREFIX = "encoder.block."
DEFAULT_TRAINABLE_BLOCKS = (
    2  # the unfrozen policy trains the last two encoder blocks (14,158,848 parameters)
)
MIN_SCORED_RECORDS = 50  # below this a scored dataset is labelled a small sample
ARTIFACT_FORMAT = "org.valcorza.moment-1-base.classifier-adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
POLICY_FROZEN = "frozen encoder + linear probe"

METRIC_DEFINITIONS = {
    "accuracy": "fraction of windows whose predicted label equals the gold label",
    "macro_f1": "unweighted mean over classes of the per-class F1 (precision-recall harmonic mean)",
    "per_class": "precision, recall, F1 and support for every class of the gold label set",
    "confusion": "rows are gold classes, columns predicted classes, in the order of `classes`",
    "log_loss": "mean negative log-probability the head assigns to the gold label (the selection signal)",
}


# ---- metrics -------------------------------------------------------------------------------------


def classification_metrics(
    y_true: Sequence[str], y_pred: Sequence[str], classes: Sequence[str]
) -> dict[str, Any]:
    """Accuracy, macro-F1, per-class precision / recall / F1 / support and the confusion matrix."""
    if len(y_true) != len(y_pred):
        raise ValueError(f"{len(y_true)} gold labels for {len(y_pred)} predictions")
    if not y_true:
        raise ValueError("at least one labelled window is required")
    index = {c: i for i, c in enumerate(classes)}
    unknown = sorted({*y_true, *y_pred} - set(index))
    if unknown:
        raise ValueError(f"labels outside the class list: {unknown}")
    confusion = [[0] * len(classes) for _ in classes]
    for gold, pred in zip(y_true, y_pred, strict=True):
        confusion[index[gold]][index[pred]] += 1
    per_class = {}
    f1s = []
    for i, name in enumerate(classes):
        tp = confusion[i][i]
        fp = sum(confusion[r][i] for r in range(len(classes))) - tp
        fn = sum(confusion[i]) - tp
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_class[name] = {"precision": precision, "recall": recall, "f1": f1, "support": tp + fn}
        f1s.append(f1)
    correct = sum(confusion[i][i] for i in range(len(classes)))
    return {
        "n": len(y_true),
        "accuracy": correct / len(y_true),
        "macro_f1": float(np.mean(f1s)),
        "per_class": per_class,
        "confusion": confusion,
        "classes": list(classes),
        "definitions": dict(METRIC_DEFINITIONS),
    }


def majority_baseline(
    train_labels: Sequence[str], test_labels: Sequence[str], classes: Sequence[str]
) -> dict[str, Any]:
    """The most frequent training label predicted for every window (the floor)."""
    counts: dict[str, int] = {}
    for label in train_labels:
        counts[label] = counts.get(label, 0) + 1
    majority = max(sorted(counts), key=lambda c: counts[c])
    out = classification_metrics(list(test_labels), [majority] * len(test_labels), classes)
    out["baseline"] = f"majority training label ({majority!r}) predicted for every window"
    return out


def knn_predict(
    train_features: np.ndarray, train_labels: Sequence[str], test_features: np.ndarray, k: int = 5
) -> list[str]:
    """Cosine k-NN vote over L2-normalised feature rows (ties broken by the nearer neighbour)."""
    if k < 1 or k > len(train_labels):
        raise ValueError(f"k must be in 1..{len(train_labels)}")
    a = np.asarray(train_features, dtype=np.float32)
    b = np.asarray(test_features, dtype=np.float32)
    a = a / np.clip(np.linalg.norm(a, axis=1, keepdims=True), 1e-12, None)
    b = b / np.clip(np.linalg.norm(b, axis=1, keepdims=True), 1e-12, None)
    sims = b @ a.T
    out = []
    for row in sims:
        order = np.argsort(-row, kind="stable")[:k]
        votes: dict[str, float] = {}
        for rank, j in enumerate(order):
            label = str(train_labels[j])
            votes[label] = votes.get(label, 0.0) + 1.0 + 1e-6 * (k - rank)
        out.append(max(votes, key=lambda c: votes[c]))
    return out


# ---- the canonical bridge and the frozen features -------------------------------------------------


def windows_for(
    records: Sequence[Mapping[str, Any]], config: MomentConfig | None = None
) -> tuple[WindowSet, list[dict[str, Any]]]:
    """Validated records → the canonical `WindowSet` through the repository's own validation and
    windowing path, with the records re-ordered to the window order (series ids sort lexically)."""
    checked = validate_dataset(records, min_records=1, max_records=MAX_RECORDS)["records"]
    config = config or MomentConfig(task="embedding")
    report, frame = validate_long_frame(records_to_long_frame(checked), config)
    windows = to_windows(frame, config, report=report, frame=frame)
    by_id = {r["id"]: r for r in checked}
    ordered = [by_id[sid] for sid in windows.series_ids]
    if len(ordered) != len(checked):
        raise ValueError("windowing dropped records; ids must be unique")
    return windows, ordered


def features(
    records: Sequence[Mapping[str, Any]], model: LoadedMoment, *, batch_size: int = 8
) -> tuple[np.ndarray, list[dict[str, Any]]]:
    """Frozen-policy features: one L2-normalised 768-d pooled embedding per validated window, through
    `embed` (eval mode, no gradient), plus the records in window order."""
    windows, ordered = windows_for(records)
    result = embed(windows, model, batch_size=batch_size, warmup=False)
    vectors = np.asarray(result.embeddings, dtype=np.float32)
    vectors = vectors / np.clip(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-12, None)
    return vectors, ordered


def knn_baseline(
    model: LoadedMoment,
    train: Sequence[Mapping[str, Any]],
    test: Sequence[Mapping[str, Any]],
    *,
    k: int = 5,
) -> dict[str, Any]:
    """Cosine k-NN vote over the frozen embeddings — what the representation gives with no training."""
    train_vectors, train_ordered = features(train, model)
    test_vectors, test_ordered = features(test, model)
    classes = sorted({r["label"] for r in train_ordered})
    predictions = knn_predict(train_vectors, [r["label"] for r in train_ordered], test_vectors, k=k)
    out = classification_metrics([r["label"] for r in test_ordered], predictions, classes)
    out["baseline"] = f"cosine {k}-NN vote over the frozen embeddings of the training windows"
    out["k"] = k
    return out


# ---- the adapter -----------------------------------------------------------------------------------


@dataclass(eq=False)
class ClassifierAdapter:
    """A trained head over the (possibly adapted) encoder plus the record of how it was chosen."""

    head: Any = field(repr=False)
    classes: list[str]
    policy: str
    config: dict[str, Any]
    history: list[dict[str, Any]] = field(default_factory=list)
    trainable_names: list[str] = field(default_factory=list)

    def summary(self) -> dict[str, Any]:
        return {"policy": self.policy, "classes": list(self.classes), **self.config}


def _trainable_names(model: LoadedMoment, trainable_blocks: int) -> list[str]:
    if (
        isinstance(trainable_blocks, bool)
        or not isinstance(trainable_blocks, int)
        or not 0 <= trainable_blocks <= ENCODER_BLOCKS
    ):
        raise ValueError(f"trainable_blocks must be an int in 0..{ENCODER_BLOCKS}")
    if trainable_blocks == 0:
        return []
    first = ENCODER_BLOCKS - trainable_blocks
    prefixes = tuple(f"{BLOCK_PREFIX}{k}." for k in range(first, ENCODER_BLOCKS))
    return [name for name, _p in model.pipeline.named_parameters() if name.startswith(prefixes)]


def _embed_batch(
    model: LoadedMoment, windows: WindowSet, index: Sequence[int], *, grad: bool
) -> Any:
    import torch

    x = torch.from_numpy(windows.x_enc[list(index)]).to(model.identity.device)
    mask = torch.from_numpy(windows.input_mask[list(index)]).to(model.identity.device)
    if grad:
        out = model.pipeline.embed(x_enc=x, input_mask=mask, reduction="mean")
    else:
        with torch.no_grad():
            out = model.pipeline.embed(x_enc=x, input_mask=mask, reduction="mean")
    return torch.nn.functional.normalize(out.embeddings.float(), dim=-1)


def _probabilities(
    model: LoadedMoment, adapter: ClassifierAdapter, windows: WindowSet, *, batch_size: int = 8
) -> Any:
    import torch

    model.pipeline.eval()
    rows = []
    with torch.no_grad():
        for start in range(0, windows.n_windows, batch_size):
            feats = _embed_batch(
                model, windows, range(start, min(start + batch_size, windows.n_windows)), grad=False
            )
            rows.append(torch.softmax(adapter.head(feats), dim=-1).cpu())
    return torch.cat(rows, dim=0)


def classify(
    model: LoadedMoment, adapter: ClassifierAdapter, records: Sequence[Mapping[str, Any]]
) -> dict[str, Any]:
    """Label validated windows with the trained head over the (possibly adapted) encoder."""
    windows, ordered = windows_for(records)
    probabilities = _probabilities(model, adapter, windows)
    return {
        "ids": [r["id"] for r in ordered],
        "labels": [adapter.classes[int(i)] for i in probabilities.argmax(dim=-1)],
        "probabilities": [[float(v) for v in row] for row in probabilities.tolist()],
        "classes": list(adapter.classes),
        "policy": adapter.policy,
        "decision_rule": "argmax of the head's softmax; not calibrated, no threshold",
        "model_id": model.identity.name,
        "model_revision": model.identity.revision,
    }


def evaluate(
    model: LoadedMoment, adapter: ClassifierAdapter, records: Sequence[Mapping[str, Any]]
) -> dict[str, Any]:
    """Score the trained head on validated labelled windows: accuracy, macro-F1, per-class, confusion, log-loss."""
    import torch

    windows, ordered = windows_for(records)
    started = time.perf_counter()
    probabilities = _probabilities(model, adapter, windows)
    gold = [r["label"] for r in ordered]
    unknown = sorted(set(gold) - set(adapter.classes))
    if unknown:
        raise ValueError(f"labels outside the head's classes: {unknown}")
    targets = torch.tensor([adapter.classes.index(g) for g in gold])
    predictions = [adapter.classes[int(i)] for i in probabilities.argmax(dim=-1)]
    metrics = classification_metrics(gold, predictions, adapter.classes)
    metrics["log_loss"] = float(
        -torch.log(probabilities[torch.arange(len(gold)), targets].clamp_min(1e-12)).mean()
    )
    metrics["policy"] = adapter.policy
    metrics["adapted"] = True
    metrics["verdict"] = "measured" if len(gold) >= MIN_SCORED_RECORDS else "measured-small-sample"
    metrics["seconds"] = round(time.perf_counter() - started, 3)
    metrics["model_id"] = model.identity.name
    metrics["model_revision"] = model.identity.revision
    return metrics


def adapt(
    model: LoadedMoment,
    train: Sequence[Mapping[str, Any]],
    val: Sequence[Mapping[str, Any]] | None = None,
    *,
    probe_steps: int = 300,
    probe_lr: float = 1e-2,
    trainable_blocks: int = DEFAULT_TRAINABLE_BLOCKS,
    epochs: int = 3,
    lr: float = 3e-4,
    batch_size: int = 8,
    seed: int = 0,
    progress: Callable[[dict[str, Any]], None] | None = None,
) -> ClassifierAdapter:
    """Two-stage bounded adaptation, selected on validation.

    Stage A (the **frozen policy**): a `Linear(768, classes)` head trained full-batch on the frozen
    L2-normalised embeddings with AdamW (`probe_lr`, weight decay 1e-4) for `probe_steps` steps — the linear
    probe — recorded as epoch 0. Stage B (the **unfrozen policy**, when `trainable_blocks` > 0 and `epochs` > 0):
    the last `trainable_blocks` T5 encoder blocks and the head trained end to end on the windows for `epochs`
    epochs (AdamW at `lr`, weight decay 0.01, gradient clipping 1.0, seeded shuffling, no augmentation); the
    patch embedding, the earlier blocks and the final norm stay frozen. Every epoch is scored on `val` and the
    epoch with the **lowest validation log-loss** is kept (epoch 0 competes) and its tensors restored; without
    `val` the final epoch is kept. The encoder inside `model.pipeline` is modified in place when the unfrozen
    policy wins — `embed` then returns different vectors for every window.
    """
    import torch

    if (
        isinstance(probe_steps, bool)
        or not isinstance(probe_steps, int)
        or not 1 <= probe_steps <= 5_000
    ):
        raise ValueError("probe_steps must be an int in 1..5000")
    if not isinstance(probe_lr, int | float) or not 0.0 < float(probe_lr) <= 1.0:
        raise ValueError("probe_lr must be in (0, 1]")
    if isinstance(epochs, bool) or not isinstance(epochs, int) or not 0 <= epochs <= 20:
        raise ValueError("epochs must be an int in 0..20")
    if not isinstance(lr, int | float) or not 0.0 < float(lr) <= 1e-2:
        raise ValueError("lr must be in (0, 1e-2]")
    if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 32:
        raise ValueError("batch_size must be an int in 1..32")
    names = _trainable_names(model, trainable_blocks)
    if model.identity.task != "embedding":
        raise ValueError("adapt() needs a pipeline loaded with task='embedding'")
    started = time.perf_counter()
    train_windows, train_ordered = windows_for(train)
    val_windows, val_ordered = windows_for(val) if val is not None else (None, None)
    classes = sorted({r["label"] for r in train_ordered})
    if len(classes) < 2:
        raise ValueError("training records need at least two classes")
    targets = torch.tensor([classes.index(r["label"]) for r in train_ordered])
    device = model.identity.device
    d_model = int(model.identity.d_model_effective)
    torch.manual_seed(seed)
    rng = random.Random(seed)

    # Stage A: the probe on frozen features.
    model.pipeline.eval()
    with torch.no_grad():
        frozen = torch.cat(
            [
                _embed_batch(
                    model,
                    train_windows,
                    range(s, min(s + batch_size, train_windows.n_windows)),
                    grad=False,
                )
                for s in range(0, train_windows.n_windows, batch_size)
            ]
        )
    head = torch.nn.Linear(d_model, len(classes)).to(device)
    probe_opt = torch.optim.AdamW(head.parameters(), lr=float(probe_lr), weight_decay=1e-4)
    probe_loss = math.nan
    for _ in range(probe_steps):
        probe_opt.zero_grad(set_to_none=True)
        loss = torch.nn.functional.cross_entropy(head(frozen.to(device)), targets.to(device))
        loss.backward()
        probe_opt.step()
        probe_loss = float(loss.detach())
    adapter = ClassifierAdapter(
        head=head, classes=classes, policy=POLICY_FROZEN, config={}, trainable_names=[]
    )

    def score() -> dict[str, float] | None:
        if val_windows is None:
            return None
        metrics = evaluate(model, adapter, val_ordered)
        return {
            "n": metrics["n"],
            "accuracy": round(metrics["accuracy"], 6),
            "macro_f1": round(metrics["macro_f1"], 6),
            "log_loss": round(metrics["log_loss"], 6),
        }

    history: list[dict[str, Any]] = [
        {
            "epoch": 0,
            "stage": "linear probe (frozen encoder)",
            "train_loss": probe_loss,
            "val": score(),
        }
    ]
    params = dict(model.pipeline.named_parameters())
    best_epoch = 0
    best_score = history[0]["val"]["log_loss"] if history[0]["val"] else math.inf
    best_state = {
        "head": {k: v.detach().clone() for k, v in head.state_dict().items()},
        "blocks": {n: params[n].detach().clone() for n in names},
    }
    policy = POLICY_FROZEN
    if names and epochs > 0:
        policy_b = f"unfrozen last {trainable_blocks} blocks + linear head"
        for p in model.pipeline.parameters():
            p.requires_grad_(False)
        for n in names:
            params[n].requires_grad_(True)
        optimiser = torch.optim.AdamW(
            [*head.parameters(), *(params[n] for n in names)], lr=float(lr), weight_decay=0.01
        )
        for epoch in range(1, epochs + 1):
            model.pipeline.train()
            order = list(range(train_windows.n_windows))
            rng.shuffle(order)
            losses = []
            for start in range(0, len(order), batch_size):
                index = order[start : start + batch_size]
                feats = _embed_batch(model, train_windows, index, grad=True)
                loss = torch.nn.functional.cross_entropy(head(feats), targets[index].to(device))
                optimiser.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    [*head.parameters(), *(params[n] for n in names)], 1.0
                )
                optimiser.step()
                losses.append(float(loss.detach()))
            model.pipeline.eval()
            adapter.policy = policy_b
            val_metrics = score()
            entry = {
                "epoch": epoch,
                "stage": policy_b,
                "train_loss": float(np.mean(losses)),
                "val": val_metrics,
            }
            history.append(entry)
            if progress is not None:
                progress(entry)
            current = (
                val_metrics["log_loss"] if val_metrics else -epoch
            )  # no val: the last epoch wins
            if current < best_score:
                best_epoch, best_score, policy = epoch, current, policy_b
                best_state = {
                    "head": {k: v.detach().clone() for k, v in head.state_dict().items()},
                    "blocks": {n: params[n].detach().clone() for n in names},
                }
        with torch.no_grad():
            head.load_state_dict(best_state["head"])
            for n, value in best_state["blocks"].items():
                params[n].copy_(value)
    for p in model.pipeline.parameters():
        p.requires_grad_(False)
    for p in head.parameters():
        p.requires_grad_(False)
    model.pipeline.eval()
    head.eval()
    adapter.policy = policy
    adapter.history = history
    adapter.trainable_names = names if policy != POLICY_FROZEN else []
    adapter.config = {
        "probe_steps": probe_steps,
        "probe_lr": float(probe_lr),
        "probe_final_loss": probe_loss,
        "trainable_blocks": trainable_blocks,
        "n_trainable_head": sum(p.numel() for p in head.parameters()),
        "n_trainable_blocks": sum(params[n].numel() for n in names),
        "n_total": sum(p.numel() for p in model.pipeline.parameters()),
        "epochs": epochs,
        "best_epoch": best_epoch,
        "selection": "lowest validation log-loss (epoch 0 = linear probe)"
        if val is not None
        else "final epoch (no validation split)",
        "lr": float(lr),
        "batch_size": batch_size,
        "n_train": train_windows.n_windows,
        "n_val": val_windows.n_windows if val_windows is not None else 0,
        "seed": seed,
        "seconds": round(time.perf_counter() - started, 3),
    }
    return adapter


# ---- artifacts ------------------------------------------------------------------------------------


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def save_artifact(
    model: LoadedMoment,
    adapter: ClassifierAdapter,
    output_dir: str | Path,
    metadata: Mapping[str, Any] | None = None,
) -> Path:
    """Write the head (and any trained encoder-block tensors) as safetensors with a manifest naming the base."""
    from safetensors.torch import save_file

    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    tensors = {
        f"head.{k}": v.detach().cpu().contiguous() for k, v in adapter.head.state_dict().items()
    }
    state = dict(model.pipeline.named_parameters())
    tensors.update({k: state[k].detach().cpu().contiguous() for k in adapter.trainable_names})
    weights_path = out / ARTIFACT_WEIGHTS_NAME
    save_file(tensors, str(weights_path), metadata={"format": "pt"})
    manifest = {
        "format": ARTIFACT_FORMAT,
        "format_version": ARTIFACT_FORMAT_VERSION,
        "base_model": {
            "id": model.identity.name,
            "revision": model.identity.revision,
            "weight_file": PINNED_WEIGHTS_FILENAME,
            "weight_sha256": model.identity.weights_sha256,
            "task": model.identity.task,
        },
        "adapter": adapter.summary(),
        "history": list(adapter.history),
        "tensors": sorted(tensors),
        "files": [
            {
                "path": ARTIFACT_WEIGHTS_NAME,
                "bytes": weights_path.stat().st_size,
                "sha256": _sha256_file(weights_path),
            }
        ],
        "metadata": dict(metadata or {}),
    }
    (out / ARTIFACT_MANIFEST_NAME).write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    return out


def load_artifact(model: LoadedMoment, artifact_dir: str | Path) -> ClassifierAdapter:
    """Verify an adapter's manifest and digest **before** deserialising, rebuild the head from the
    manifest's classes and overlay any encoder-block tensors onto `model.pipeline`."""
    root = Path(artifact_dir)
    manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
    if manifest.get("format") != ARTIFACT_FORMAT:
        raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
    base = manifest.get("base_model", {})
    if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (
        model.identity.name,
        model.identity.revision,
        model.identity.weights_sha256,
    ):
        raise ValueError(
            "artifact was adapted from a different base model, revision or weight file"
        )
    if model.identity.task != "embedding":
        raise ValueError("load_artifact() needs a pipeline loaded with task='embedding'")
    entry = manifest["files"][0]
    weights_path = root / entry["path"]
    if not weights_path.is_file():
        raise FileNotFoundError(f"artifact weights missing: {weights_path}")
    if (
        _sha256_file(weights_path) != entry["sha256"]
        or weights_path.stat().st_size != entry["bytes"]
    ):
        raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
    classes = list(manifest.get("adapter", {}).get("classes") or [])
    if len(classes) < 2:
        raise ValueError("artifact manifest does not name at least two classes")
    import torch
    from safetensors.torch import load_file

    tensors = load_file(str(weights_path))
    if sorted(tensors) != manifest["tensors"]:
        raise ValueError("artifact tensor names differ from its manifest")
    d_model = int(model.identity.d_model_effective)
    if (
        tuple(tensors.get("head.weight", torch.empty(0)).shape) != (len(classes), d_model)
        or "head.bias" not in tensors
    ):
        raise ValueError("artifact head does not match d_model and the manifest's classes")
    params = dict(model.pipeline.named_parameters())
    block_tensors = {k: v for k, v in tensors.items() if not k.startswith("head.")}
    for key, value in block_tensors.items():
        if key not in params or not key.startswith(BLOCK_PREFIX):
            raise ValueError(
                f"artifact tensor {key} is not an adaptable encoder-block tensor of the base"
            )
        if tuple(value.shape) != tuple(params[key].shape):
            raise ValueError(
                f"artifact tensor {key}: shape {tuple(value.shape)} != {tuple(params[key].shape)}"
            )
    head = torch.nn.Linear(d_model, len(classes))
    head.load_state_dict(
        {"weight": tensors["head.weight"].float(), "bias": tensors["head.bias"].float()}
    )
    head = head.to(model.identity.device).eval()
    for p in head.parameters():
        p.requires_grad_(False)
    with torch.no_grad():
        for key, value in block_tensors.items():
            params[key].copy_(value.to(params[key].dtype))
    model.pipeline.eval()
    summary = dict(manifest["adapter"])
    policy = summary.pop("policy")
    summary.pop("classes", None)
    return ClassifierAdapter(
        head=head,
        classes=classes,
        policy=policy,
        config=summary,
        history=list(manifest.get("history", [])),
        trainable_names=sorted(block_tensors),
    )

**Module 10/12:** `src/moment_pipeline/imputation.py` (carried verbatim; see the note above)

In [ ]:
"""Task 2 — reconstruction plus the v1 user-facing imputation contract.

MOMENT-1-base exposes pretrained reconstruction weights. DIMER uses that primitive
for imputation but keeps the product semantics explicit:

* canonical ``x_enc`` is finite before model entry; raw NaN never reaches MOMENT;
* the model mask is always explicit and patch-quantized;
* **default imputed output preserves every observed source value** and replaces
  only source-missing values or points the caller deliberately hid;
* full reconstruction remains separately available;
* artificial-mask evaluation scores only deliberately hidden points for which
  ground truth existed in the source. Source-missing points are never invented
  into the denominator.
"""

from __future__ import annotations

import time
from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd

# standalone rewrite (build_notebook.py): `from .canonical import (` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .embedding import TaskMismatchError` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .model import LoadedMoment` removed — names are kernel globals defined by the carried modules


class DegenerateMaskError(ValueError):
    """A window would be handed to MOMENT with nothing visible at all."""


@dataclass(frozen=True)
class ImputationMetrics:
    """Masked-point-only evaluation for deliberately hidden ground-truth points."""

    n: int
    mae: float
    rmse: float


@dataclass(frozen=True)
class ReconstructionResult:
    """Reconstruction, imputed series, masks and accounting for one call.

    ``masked_point_fraction`` is source missingness. ``model_masked_point_fraction``
    is the channel-collapsed fraction hidden from MOMENT after source missingness and
    any caller mask are combined. ``requested_visible_mask`` records only the caller's
    extra mask before patch expansion, so artificial-mask evaluation can distinguish
    requested targets from neighbouring points hidden solely because MOMENT works by
    patches.

    The final five fields have defaults for backwards compatibility with tests that
    construct a minimal result only to exercise provenance serialization.
    """

    reconstruction: np.ndarray
    point_mask: np.ndarray
    model_mask: np.ndarray
    patch_mask: np.ndarray
    masked_point_fraction: float
    masked_point_count: int
    model_masked_point_fraction: float
    model_masked_point_count: int
    masked_patch_fraction: float
    masked_patch_count: int
    series_ids: tuple[str, ...]
    window_ids: tuple[str, ...]
    channels: tuple[str, ...]
    latency_seconds: float
    input_values: np.ndarray | None = None
    input_mask: np.ndarray | None = None
    timestamps: tuple[np.ndarray, ...] = ()
    requested_visible_mask: np.ndarray | None = None
    imputed: np.ndarray | None = None

    def _require_export_state(self) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        if (
            self.input_values is None
            or self.input_mask is None
            or self.requested_visible_mask is None
            or self.imputed is None
        ):
            raise ValueError(
                "this ReconstructionResult was constructed without the v1 imputation export "
                "state; call reconstruct()/impute() rather than instantiating it manually"
            )
        return (
            self.input_values,
            self.input_mask,
            self.requested_visible_mask,
            self.imputed,
        )

    def to_frame(self) -> pd.DataFrame:
        """Return one row per non-padded (window, channel, timestamp) cell.

        ``original_value`` is NaN only where the source itself was missing. A
        deliberately hidden point retains its ground truth in ``original_value`` so
        tutorial/evaluation code can audit what was withheld, while ``imputed_value``
        contains the model-derived replacement at that point.
        """

        input_values, input_mask, requested_visible, imputed = self._require_export_state()
        rows: list[dict[str, Any]] = []
        for window_index, (series_id, window_id) in enumerate(
            zip(self.series_ids, self.window_ids, strict=True)
        ):
            stamps = self.timestamps[window_index]
            for channel_index, channel in enumerate(self.channels):
                for position in np.flatnonzero(input_mask[window_index] == 1):
                    source_observed = bool(self.point_mask[window_index, channel_index, position])
                    requested_hidden = bool(requested_visible[window_index, position] == 0)
                    rows.append(
                        {
                            "series_id": series_id,
                            "window_id": window_id,
                            "timestamp": pd.Timestamp(stamps[position]),
                            "channel": channel,
                            "original_value": (
                                float(input_values[window_index, channel_index, position])
                                if source_observed
                                else np.nan
                            ),
                            "imputed_value": float(imputed[window_index, channel_index, position]),
                            "reconstruction": float(
                                self.reconstruction[window_index, channel_index, position]
                            ),
                            "source_observed": source_observed,
                            "source_missing": not source_observed,
                            "requested_hidden": requested_hidden,
                            "model_hidden": bool(self.model_mask[window_index, position] == 0),
                        }
                    )
        return pd.DataFrame(rows)


def mask_accounting(
    windows: WindowSet, visible_points: np.ndarray, patch_mask: np.ndarray
) -> dict[str, float | int]:
    """Every reported masking number for one reconstruction call, in one place."""

    valid_points = np.broadcast_to(windows.input_mask[:, None, :], windows.point_mask.shape)
    valid_patches = to_patch_view(windows.input_mask, windows.patch_length)
    return {
        "masked_point_fraction": missing_point_fraction(windows.point_mask, windows.input_mask),
        "masked_point_count": int(((valid_points == 1) & (windows.point_mask == 0)).sum()),
        "model_masked_point_fraction": hidden_position_fraction(
            visible_points, windows.input_mask
        ),
        "model_masked_point_count": int(
            ((windows.input_mask == 1) & (visible_points == 0)).sum()
        ),
        "masked_patch_fraction": masked_patch_fraction_of(
            patch_mask, windows.input_mask, windows.patch_length
        ),
        "masked_patch_count": int(((valid_patches == 1) & (patch_mask == 0)).sum()),
    }


def _requested_visible_mask(windows: WindowSet, mask: np.ndarray | None) -> np.ndarray:
    """Normalize only the caller's extra mask, independent of source missingness."""

    if mask is None:
        return np.ones_like(windows.input_mask, dtype=np.float32)
    requested = np.asarray(mask, dtype=np.float32)
    if requested.ndim == 3:
        if requested.shape != windows.point_mask.shape:
            raise ValueError(
                f"mask shape {requested.shape} != point_mask shape {windows.point_mask.shape}"
            )
        # MOMENT has no channel axis in its reconstruction mask. If one channel is
        # deliberately hidden at a timestamp, the model must treat that timestamp as
        # hidden for every channel; record the same conservative collapse explicitly.
        requested = requested.min(axis=1)
    if requested.shape != windows.input_mask.shape:
        raise ValueError(f"mask shape {requested.shape} != expected {windows.input_mask.shape}")
    if not np.isin(np.unique(requested), (0.0, 1.0)).all():
        raise ValueError("mask must be binary (1 = visible to the model, 0 = hidden)")
    return requested.astype(np.float32)


def _combine_masks(windows: WindowSet, requested_visible: np.ndarray) -> np.ndarray:
    """Visible-point mask: source observed in every channel AND caller-visible."""

    return np.minimum(windows.model_point_mask, requested_visible).astype(np.float32)


def masked_point_metrics(result: ReconstructionResult) -> ImputationMetrics:
    """Evaluate only deliberately hidden source-observed points.

    Source missing values have no truth and are excluded. Positions hidden only by
    patch expansion are also excluded: the tutorial metric answers how well the model
    reconstructed the points the evaluator intentionally held out, not every neighbour
    MOMENT had to hide internally to satisfy its patch contract.
    """

    input_values, input_mask, requested_visible, imputed = result._require_export_state()
    valid = np.broadcast_to(input_mask[:, None, :], result.point_mask.shape) == 1
    deliberate = np.broadcast_to(
        requested_visible[:, None, :] == 0, result.point_mask.shape
    )
    scorable = valid & (result.point_mask == 1) & deliberate
    n = int(scorable.sum())
    if n == 0:
        raise ValueError(
            "no deliberately hidden source-observed points are available for imputation "
            "evaluation; pass an artificial mask to reconstruct()/impute()"
        )
    error = imputed[scorable] - input_values[scorable]
    return ImputationMetrics(
        n=n,
        mae=float(np.mean(np.abs(error))),
        rmse=float(np.sqrt(np.mean(np.square(error)))),
    )


def reconstruct(
    windows: WindowSet,
    model: LoadedMoment,
    mask: np.ndarray | None = None,
    batch_size: int = 8,
    warmup: bool = True,
) -> ReconstructionResult:
    """Run the pinned reconstruction path and construct the v1 imputed product.

    Args:
        windows: canonical windows; ``x_enc`` is finite by construction.
        model: pipeline loaded with ``task='reconstruction'``.
        mask: optional extra hiding mask, ``(batch, seq_len)`` or
            ``(batch, channels, seq_len)``. One means visible. It is combined with
            source missingness and then patch-quantized for MOMENT.
        warmup: run one discarded forward pass before timing the scored call.
    """

    import torch

    if model.identity.task != "reconstruction":
        raise TaskMismatchError(
            f"reconstruct() needs a pipeline loaded with task='reconstruction', got "
            f"{model.identity.task!r}"
        )
    if not np.isfinite(windows.x_enc).all():
        raise ValueError(
            "x_enc contains non-finite values; MOMENT.reconstruct has no nan_to_num and "
            "would propagate them. Build windows through moment_pipeline.canonical."
        )

    requested_visible = _requested_visible_mask(windows, mask)
    visible_points = _combine_masks(windows, requested_visible)
    patch_mask = to_patch_view(visible_points, windows.patch_length)
    model_mask = expand_patch_view(patch_mask, windows.patch_length)

    blind = np.flatnonzero(
        (patch_mask.sum(axis=1) == 0) & (windows.input_mask.sum(axis=1) > 0)
    )
    if blind.size:
        raise DegenerateMaskError(
            "[DEGENERATE_MASK] no patch is visible in window(s) "
            f"{[windows.window_ids[i] for i in blind.tolist()]}; MOMENT would normalize "
            "against an all-zero mask and return NaN. This is a masking fault, not a "
            "pre-fill fault: every value in x_enc is finite."
        )

    device = model.identity.device
    x_all = torch.from_numpy(windows.x_enc)
    input_all = torch.from_numpy(windows.input_mask)
    mask_all = torch.from_numpy(model_mask)

    def _run() -> np.ndarray:
        chunks: list[np.ndarray] = []
        with torch.no_grad():
            for start in range(0, windows.n_windows, batch_size):
                stop = start + batch_size
                out = model.pipeline.reconstruct(
                    x_enc=x_all[start:stop].to(device),
                    input_mask=input_all[start:stop].to(device),
                    mask=mask_all[start:stop].to(device),
                )
                chunks.append(out.reconstruction.detach().float().cpu().numpy())
        return np.concatenate(chunks, axis=0)

    if warmup:
        _run()
    started = time.perf_counter()
    reconstruction = _run()
    latency = time.perf_counter() - started

    if not np.isfinite(reconstruction).all():
        raise ValueError(
            "reconstruction contains non-finite values despite finite pre-fill; "
            "this violates the RFC M-2 contract"
        )

    # Replacement semantics are intentionally narrower than model_mask. A source
    # missing value is replaced. A source-observed value is replaced only when the
    # caller deliberately requested it hidden. Neighbouring observed points that MOMENT
    # had to hide merely because they share an 8-step patch remain untouched.
    valid_cells = np.broadcast_to(
        windows.input_mask[:, None, :] == 1, windows.point_mask.shape
    )
    deliberate_cells = np.broadcast_to(
        requested_visible[:, None, :] == 0, windows.point_mask.shape
    )
    replacement = valid_cells & ((windows.point_mask == 0) | deliberate_cells)
    imputed = np.where(replacement, reconstruction, windows.x_enc).astype(np.float32)

    return ReconstructionResult(
        reconstruction=reconstruction,
        point_mask=windows.point_mask,
        model_mask=model_mask,
        patch_mask=patch_mask,
        series_ids=windows.series_ids,
        window_ids=windows.window_ids,
        channels=windows.channels,
        latency_seconds=latency,
        input_values=windows.x_enc,
        input_mask=windows.input_mask,
        timestamps=windows.timestamps,
        requested_visible_mask=requested_visible,
        imputed=imputed,
        **mask_accounting(windows, visible_points, patch_mask),
    )


def impute(
    windows: WindowSet,
    model: LoadedMoment,
    mask: np.ndarray | None = None,
    batch_size: int = 8,
    warmup: bool = True,
) -> ReconstructionResult:
    """User-facing alias for :func:`reconstruct` emphasizing imputed output semantics."""

    return reconstruct(
        windows,
        model,
        mask=mask,
        batch_size=batch_size,
        warmup=warmup,
    )

**Module 11/12:** `src/moment_pipeline/anomaly.py` (carried verbatim; see the note above)

In [ ]:
"""Task 3 — reconstruction-based anomaly **scoring**.

Phase 4. The primitive is the per-element reconstruction residual under a named loss,
shape `(batch, channel, timestep)`, and that is the whole product: a score, not a verdict.

What the RFC fixes, and what this module therefore fixes:

* **Raw scores are mandatory output.** `AnomalyResult.anomaly_score` is the residual
  itself. Nothing here converts it into a boolean.
* **There is no universal binary threshold in v1**, so this module ships none — not as a
  default, not as a keyword argument, not as a constant. Calibration belongs to a caller
  who owns a reference segment, and `MODEL_CARD.md` says why a shipped default would be
  indefensible on an unmasked self-reconstruction residual.
* **Channel aggregation is DIMER-owned, explicit and recorded.** `"none"` is the default
  and the safest mode; `"mean"` and `"max"` collapse the channel axis and nothing else.
* **The score is an unmasked self-reconstruction residual** (RFC M-6): the model
  reconstructs a window it can see, and the residual is scored. It is not a forecast
  residual, and a drift the model reconstructs faithfully scores low by construction.

The scored domain is the sharp edge, and it is narrower than "every position":

1. **Padding is never scored.** Left-padding is not data.
2. **Pre-filled positions are never scored.** `x_enc` there holds `prefill_value`, a
   fabricated number; `|reconstruction - 0.0|` says something about the sentinel, not
   about the series.
3. **Positions the model could not see are never scored.** The mask is patch-quantized,
   so one missing point hides its whole 8-step patch (RFC M-3). The residual at the seven
   observed neighbours is a genuine number — but it is an *imputation* residual, produced
   from a patch the encoder was shown as unobserved, and it is systematically larger than
   a self-reconstruction residual. Mixing the two into one column would put a spike at
   every gap edge and invite the reader to call it an anomaly.

So `scored_mask` is `input_mask AND model_mask`, and every value in `anomaly_score` is the
same quantity. Unscored positions are `NaN` — a deliberate "not defined here" marker, not
a propagated NaN — and are counted by reason on the result. `reconstruction_error` keeps
the raw finite residual at every position for debugging; it is not the export column.
"""

from __future__ import annotations

import dataclasses
import time
from dataclasses import dataclass, field
from typing import Any

import numpy as np
import pandas as pd

# standalone rewrite (build_notebook.py): `from .canonical import WindowSet` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .imputation import ReconstructionResult, reconstruct` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .model import LoadedMoment` removed — names are kernel globals defined by the carried modules

#: Elementwise losses. Names match the upstream vocabulary of
#: `momentfm.utils.utils.get_anomaly_criterion` ("mae" -> L1Loss, "mse" -> MSELoss), both
#: with `reduction="none"`, which is what makes the score per-element.
ANOMALY_LOSSES = ("mae", "mse")
DEFAULT_LOSS = "mae"

#: Channel aggregation. "none" is the default: it is the only mode that loses nothing.
CHANNEL_AGGREGATIONS = ("none", "mean", "max")
DEFAULT_AGGREGATION = "none"

SCORE_POLICY = (
    "unmasked self-reconstruction residual: the model reconstructs a window it can see in "
    "full and the residual is scored. NOT a forecast residual -- a drift the model "
    "reconstructs faithfully scores low by construction. Scores are raw and uncalibrated; "
    "this pipeline ships no binary threshold, and the residual scale depends on the "
    "series, so a threshold fitted on one series does not transfer to another"
)

SCORED_DOMAIN_POLICY = (
    "scored only where the position is non-padded AND was visible to the model. Padding "
    "is not data; pre-filled positions would score the sentinel rather than the series; "
    "and a position inside a patch the mask hid yields an imputation residual, which is "
    "systematically larger than a self-reconstruction residual and would read as a spike "
    "at every gap edge. Unscored positions are NaN by construction, not by propagation"
)


class AnomalyConfigError(ValueError):
    """An unsupported loss or channel aggregation was requested."""


@dataclass(frozen=True)
class AnomalyResult:
    """Raw reconstruction residuals plus the accounting needed to read them.

    `anomaly_score` is the export column: `NaN` wherever the score is not defined (see
    `SCORED_DOMAIN_POLICY`). `reconstruction_error` is the same residual computed
    everywhere and left finite, for callers debugging the pre-fill or the mask. With
    `channel_aggregation="none"` the two share the shape `(batch, channel, timestep)` and
    differ only in those NaNs.
    """

    reconstruction: np.ndarray
    reconstruction_error: np.ndarray
    anomaly_score: np.ndarray
    scored_mask: np.ndarray
    point_mask: np.ndarray
    model_mask: np.ndarray
    series_ids: tuple[str, ...]
    window_ids: tuple[str, ...]
    channels: tuple[str, ...]
    loss: str
    channel_aggregation: str
    latency_seconds: float
    #: Cells the score is defined at, and the two reasons the rest were dropped. Counted
    #: over non-padded (window, channel, position) cells, so the three add up to
    #: `input_mask.sum() * n_channels`.
    scored_point_count: int
    unscored_prefilled_count: int
    unscored_hidden_by_patch_count: int
    scored_point_fraction: float
    masked_point_fraction: float
    masked_patch_fraction: float
    score_policy: str = SCORE_POLICY
    scored_domain_policy: str = SCORED_DOMAIN_POLICY
    #: There is no threshold in v1. The field states the absence so an export cannot be
    #: read as "thresholded with the default".
    threshold_policy: str = "none applied; raw scores only (RFC Task 3)"
    timestamps: tuple[np.ndarray, ...] = field(repr=False, default=())

    @property
    def is_aggregated(self) -> bool:
        return self.channel_aggregation != "none"

    def to_frame(self) -> pd.DataFrame:
        """`series_id, timestamp, channel, reconstruction, reconstruction_error,
        anomaly_score` — the RFC Task 3 output, one row per non-padded position.

        Padded positions are dropped: they carry no timestamp (`NaT`) and belong to no
        series. Unscored-but-real positions are kept with `anomaly_score = NaN` and
        `scored = False`, because silently dropping them would hide exactly the gaps a
        reader needs to see.

        Under a channel aggregation the frame carries `channel = <aggregation name>` and
        **omits `reconstruction` and `reconstruction_error`**: those are per-channel
        quantities and collapsing them would invent a number. Only the score aggregates,
        which is the whole point of the aggregation being score-only.
        """
        frames: list[pd.DataFrame] = []
        for w, window_id in enumerate(self.window_ids):
            stamps = self.timestamps[w] if self.timestamps else None
            keep = (
                np.flatnonzero(~pd.isna(stamps))
                if stamps is not None
                else np.arange(self.reconstruction.shape[2])
            )
            stamp_column = stamps[keep] if stamps is not None else keep
            common = {"series_id": self.series_ids[w], "window_id": window_id}
            if self.is_aggregated:
                frames.append(
                    pd.DataFrame(
                        {
                            **common,
                            "timestamp": stamp_column,
                            "channel": self.channel_aggregation,
                            "anomaly_score": self.anomaly_score[w, keep],
                            "scored": self.scored_mask[w, keep].astype(bool),
                        }
                    )
                )
                continue
            for c, channel in enumerate(self.channels):
                frames.append(
                    pd.DataFrame(
                        {
                            **common,
                            "timestamp": stamp_column,
                            "channel": channel,
                            "reconstruction": self.reconstruction[w, c, keep],
                            "reconstruction_error": self.reconstruction_error[w, c, keep],
                            "anomaly_score": self.anomaly_score[w, c, keep],
                            "scored": self.scored_mask[w, c, keep].astype(bool),
                        }
                    )
                )
        return pd.concat(frames, ignore_index=True)


def residual(x_enc: np.ndarray, reconstruction: np.ndarray, loss: str = DEFAULT_LOSS) -> np.ndarray:
    """Elementwise reconstruction residual under `loss`. Pure; no model, no weights.

    `"mae"` is `|x - x_hat|` and `"mse"` is `(x - x_hat)**2`, matching
    `get_anomaly_criterion`'s `L1Loss(reduction="none")` / `MSELoss(reduction="none")`.
    """
    if loss not in ANOMALY_LOSSES:
        raise AnomalyConfigError(
            f"loss={loss!r} is not supported; choose one of {list(ANOMALY_LOSSES)}"
        )
    difference = reconstruction.astype(np.float64) - x_enc.astype(np.float64)
    error = np.abs(difference) if loss == "mae" else np.square(difference)
    return error.astype(np.float32)


def aggregate_channels(score: np.ndarray, how: str = DEFAULT_AGGREGATION) -> np.ndarray:
    """Collapse the channel axis of a `(batch, channel, timestep)` score, and nothing else.

    NaN-aware: a position aggregates over the channels whose score is defined, and stays
    NaN only where no channel scored. `"none"` returns the input untouched.
    """
    if how not in CHANNEL_AGGREGATIONS:
        raise AnomalyConfigError(
            f"channel_aggregation={how!r} is not supported; choose one of "
            f"{list(CHANNEL_AGGREGATIONS)}"
        )
    if how == "none":
        return score
    defined = ~np.isnan(score)
    all_nan = ~defined.any(axis=1)
    # `nanmean`/`nanmax` warn on an all-NaN slice and this is a routine case -- a position
    # unscored in every channel -- so the reduction is done on a filled copy with a
    # neutral identity and the all-NaN positions are restored afterwards. No warning, and
    # the identity never reaches the output.
    if how == "mean":
        counts = defined.sum(axis=1)
        collapsed = np.where(defined, score, 0.0).sum(axis=1) / np.maximum(counts, 1)
    else:
        collapsed = np.where(defined, score, -np.inf).max(axis=1)
    return np.where(all_nan, np.nan, collapsed).astype(np.float32)


def score_anomalies(
    windows: WindowSet,
    model: LoadedMoment,
    loss: str = DEFAULT_LOSS,
    channel_aggregation: str = DEFAULT_AGGREGATION,
    batch_size: int = 8,
    warmup: bool = True,
) -> AnomalyResult:
    """Score every window by its unmasked self-reconstruction residual.

    Runs the same pinned reconstruction entry point as `imputation.reconstruct` with no
    extra hiding mask, so the model sees every observed point — including the ones it is
    being scored on. That is the RFC's v1 definition, and `MODEL_CARD.md` records what it
    costs.

    Args:
        loss: `"mae"` (default) or `"mse"`; recorded on the result and in provenance.
        channel_aggregation: `"none"` (default), `"mean"` or `"max"`. Collapses the
            channel axis only.
        warmup: as `imputation.reconstruct` — a discarded forward pass so
            `latency_seconds` excludes lazy allocation. Pass False in production.

    Raises:
        AnomalyConfigError: unsupported `loss` or `channel_aggregation`, refused before
            the model runs rather than after a forward pass.
    """
    if loss not in ANOMALY_LOSSES:
        raise AnomalyConfigError(
            f"loss={loss!r} is not supported; choose one of {list(ANOMALY_LOSSES)}"
        )
    if channel_aggregation not in CHANNEL_AGGREGATIONS:
        raise AnomalyConfigError(
            f"channel_aggregation={channel_aggregation!r} is not supported; choose one of "
            f"{list(CHANNEL_AGGREGATIONS)}"
        )

    started = time.perf_counter()
    # mask=None is the unmasked self-reconstruction: the only positions hidden are the
    # ones the source did not contain, which `reconstruct` hides for us and which the
    # scored domain then excludes anyway.
    reconstructed = reconstruct(
        windows, model, mask=None, batch_size=batch_size, warmup=warmup
    )
    result = score_from_reconstruction(
        windows, reconstructed, loss=loss, channel_aggregation=channel_aggregation
    )
    return _with_latency(result, time.perf_counter() - started)


def score_from_reconstruction(
    windows: WindowSet,
    reconstructed: ReconstructionResult,
    loss: str = DEFAULT_LOSS,
    channel_aggregation: str = DEFAULT_AGGREGATION,
) -> AnomalyResult:
    """Derive scores from a reconstruction that has already been computed.

    Split out from `score_anomalies` so the scoring arithmetic — the scored domain, the
    residual, the aggregation — is testable against a `WindowSet` without 454 MB of
    weights, and so a caller who already holds a `ReconstructionResult` does not pay for a
    second forward pass.
    """
    error = residual(windows.x_enc, reconstructed.reconstruction, loss)

    # Non-padded AND visible to the model. `model_mask` is the patch-expanded mask that
    # `reconstruct` actually handed to momentfm, so this is what the encoder saw, not what
    # the caller hoped it saw.
    visible = windows.input_mask * reconstructed.model_mask
    scored = np.broadcast_to(visible[:, None, :], error.shape).astype(np.float32)

    valid = np.broadcast_to(windows.input_mask[:, None, :], error.shape)
    prefilled = (valid == 1) & (windows.point_mask == 0)
    hidden_but_observed = (valid == 1) & (windows.point_mask == 1) & (scored == 0)

    score = np.where(scored == 1, error, np.nan).astype(np.float32)
    score = aggregate_channels(score, channel_aggregation)
    scored_mask = visible.astype(np.float32) if channel_aggregation != "none" else scored

    denominator = float(valid.sum())
    return AnomalyResult(
        reconstruction=reconstructed.reconstruction,
        reconstruction_error=error,
        anomaly_score=score,
        scored_mask=scored_mask,
        point_mask=windows.point_mask,
        model_mask=reconstructed.model_mask,
        series_ids=windows.series_ids,
        window_ids=windows.window_ids,
        channels=windows.channels,
        loss=loss,
        channel_aggregation=channel_aggregation,
        latency_seconds=reconstructed.latency_seconds,
        scored_point_count=int((scored == 1).sum()),
        unscored_prefilled_count=int(prefilled.sum()),
        unscored_hidden_by_patch_count=int(hidden_but_observed.sum()),
        scored_point_fraction=(float((scored == 1).sum()) / denominator) if denominator else 0.0,
        masked_point_fraction=reconstructed.masked_point_fraction,
        masked_patch_fraction=reconstructed.masked_patch_fraction,
        timestamps=windows.timestamps,
    )


def _with_latency(result: AnomalyResult, seconds: float) -> AnomalyResult:
    """Total wall time for `score_anomalies`, which is the reconstruction plus the
    scoring arithmetic — not the reconstruction alone."""
    return dataclasses.replace(result, latency_seconds=seconds)


__all__ = [
    "ANOMALY_LOSSES",
    "CHANNEL_AGGREGATIONS",
    "DEFAULT_AGGREGATION",
    "DEFAULT_LOSS",
    "AnomalyConfigError",
    "AnomalyResult",
    "aggregate_channels",
    "residual",
    "score_anomalies",
    "score_from_reconstruction",
    "top_k_recall",
]


def top_k_recall(
    scores: pd.DataFrame, labels: pd.DataFrame, *, channel: str | None = None
) -> dict[str, Any]:
    """The tutorial's ranking metric: recall of the labelled anomalies within the top-k of
    the raw-score ranking, where k is the number of labelled anomalies on `channel`.

    `scores` is `AnomalyResult.to_frame()`; `labels` carries `series_id`, `timestamp` and a
    boolean `is_injected_anomaly`. Only scored positions rank. Raises `ValueError` when no
    position is scored or the labels mark no anomaly among the scored positions. Returns the
    value plus the ranked frame and the labelled ranks so a reader can inspect the ordering.
    """
    if channel is None:
        present = set(scores["channel"])
        channel = "vibration" if "vibration" in present else str(scores["channel"].iloc[0])
    ranked = (
        scores[(scores["channel"] == channel) & scores["scored"]]
        .copy()
        .sort_values("anomaly_score", ascending=False, kind="mergesort")
        .reset_index(drop=True)
    )
    if ranked.empty:
        raise ValueError(
            "No scored positions are available after masking/padding; provide a series with "
            "observed values."
        )
    ranked["rank"] = ranked.index + 1
    marks = labels.copy()
    marks["timestamp"] = pd.to_datetime(marks["timestamp"])
    ranked = ranked.merge(marks, on=["series_id", "timestamp"], how="left")
    ranked["is_injected_anomaly"] = ranked["is_injected_anomaly"].fillna(False).astype(bool)
    k = int(ranked["is_injected_anomaly"].sum())
    if k == 0:
        raise ValueError("the labels mark no anomaly among the scored positions of " + channel)
    hits = int(ranked.head(k)["is_injected_anomaly"].sum())
    return {
        "k": k,
        "channel": channel,
        "value": hits / k,
        "injected_ranks": ranked.loc[
            ranked["is_injected_anomaly"], ["timestamp", "rank", "anomaly_score"]
        ].to_dict("records"),
        "ranked": ranked,
    }

**Module 12/12:** `src/moment_pipeline/roles.py` (carried verbatim; see the note above)

In [ ]:
"""Role-stage helpers shared by the three tutorials (DIMER NOTEBOOK_SPEC 1.1 DAT24 / EVAL21).

`validate_inputs` is the public validation stage: it routes the long frame through exactly
the calls the task functions rely on (`validate_long_frame` then `to_windows`), so it raises
exactly what canonicalisation would raise, and reports what was proven as an **input
manifest**. `evaluation_report` is the public evaluation stage: it adds no metric of its own —
imputation evidence comes from `masked_point_metrics`, anomaly evidence from `top_k_recall`,
and embeddings are representations with no correctness metric — and always produces a report,
saying what would make the task measurable when nothing is.
"""

from __future__ import annotations

from typing import Any

import numpy as np
import pandas as pd

# standalone rewrite (build_notebook.py): `from .anomaly import AnomalyResult, top_k_recall` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .canonical import WindowSet, to_windows` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .config import PATCH_LENGTH, SEQUENCE_LENGTH, MomentConfig` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .embedding import EmbeddingResult` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .imputation import ReconstructionResult, masked_point_metrics` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .model import PINNED_MODEL_ID, PINNED_REVISION, LoadedMoment, ModelIdentity` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .validation import REQUIRED_COLUMNS, ValidationError, validate_long_frame` removed — names are kernel globals defined by the carried modules

__all__ = ["INPUT_SCHEMA", "validate_inputs", "evaluation_report"]

_DEFAULT_LIMITS = MomentConfig().limits

#: The input contract and every named ceiling, in one readable structure.
INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "long-format pandas DataFrame with columns "
        + ", ".join(REQUIRED_COLUMNS)
        + "; one row per (series_id, channel, timestamp)"
    ),
    "timestamps": (
        "parseable and strictly increasing per (series_id, channel); irregular spacing is "
        "reported per series and refused when strict_frequency is set"
    ),
    "values": "numeric; NaN marks a source-missing point, which never reaches the model",
    "sequence_length": SEQUENCE_LENGTH,
    "patch_length": PATCH_LENGTH,
    "max_rows": _DEFAULT_LIMITS.max_rows,
    "max_series": _DEFAULT_LIMITS.max_series,
    "max_channels": _DEFAULT_LIMITS.max_channels,
    "max_windows": _DEFAULT_LIMITS.max_windows,
    "canonicalisation": (
        "one window per series: the final sequence_length timestamps are kept (longer series "
        "are truncated, shorter ones left-padded and the padding excluded by the input mask)"
    ),
}


def validate_inputs(
    df: pd.DataFrame,
    config: MomentConfig | None = None,
    *,
    names: list[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-window observations, verdict).

    Rejection is reported by raising exactly as the task path would — `validate_long_frame`
    followed by `to_windows`, the same two calls every tutorial and task function makes — so a
    caller that wants the finding recorded catches `ValidationError` and stores `str(exc)`
    under `findings`.
    """
    config = config or MomentConfig()
    report, frame = validate_long_frame(df, config)
    windows: WindowSet = to_windows(frame, config, report=report, frame=frame)
    if names is not None and len(names) != windows.n_windows:
        raise ValidationError(
            "NAMES_LENGTH_MISMATCH",
            f"names must have one entry per window: got {len(names)} for {windows.n_windows}",
            {"n_names": len(names), "n_windows": windows.n_windows},
        )
    inputs: list[dict[str, Any]] = []
    for index, window_id in enumerate(windows.window_ids):
        valid = int(windows.input_mask[index].sum())
        inputs.append(
            {
                "id": names[index] if names else str(window_id),
                "series_id": str(windows.series_ids[index]),
                "valid_positions": valid,
                "padded": bool(windows.padded[index]),
                "truncated": bool(windows.truncated[index]),
            }
        )
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": inputs,
        "task": config.task,
        "n_rows": int(report.n_rows),
        "n_series": len(report.series_ids),
        "n_channels": len(report.channels),
        "channels": list(windows.channels),
        "irregular_series": list(report.irregular_series),
        "window_shape": list(windows.x_enc.shape),
        "masked_point_fraction": float(windows.masked_point_fraction),
        "masked_patch_fraction": float(windows.masked_patch_fraction),
        "verdict": "accepted",
        "findings": [],
        "model_id": PINNED_MODEL_ID,
        "model_revision": PINNED_REVISION,
    }


def _identity_block(model: LoadedMoment | ModelIdentity | None) -> dict[str, Any]:
    identity = getattr(model, "identity", model)
    return {
        "model_id": getattr(identity, "name", None),
        "model_revision": getattr(identity, "revision", None),
    }


def _not_measurable(base: dict[str, Any], reason: str, needs: str) -> dict[str, Any]:
    return {
        **base,
        "metrics": [],
        "baselines": [],
        "verdict": "not-measurable",
        "reason": reason,
        "needs": needs,
    }


def evaluation_report(
    result: EmbeddingResult | ReconstructionResult | AnomalyResult,
    labels: pd.DataFrame | None = None,
    *,
    model: LoadedMoment | ModelIdentity | None = None,
    sample_kind: str = "synthetic",
    channel: str | None = None,
    baseline: dict[str, float] | None = None,
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    - `EmbeddingResult`: embeddings are representations, not predictions — the verdict is
      always `not-measurable` and the report says what downstream task labels would score them.
    - `ReconstructionResult`: when the result carries a deliberate artificial mask,
      `masked_point_metrics` (MAE/RMSE on the deliberately hidden, source-observed points) is
      reported with the verdict `sample-sanity`; an optional tutorial `baseline`
      (`{"mae": ..., "rmse": ...}`, e.g. linear interpolation) is carried under `baselines`.
      Without a deliberate mask nothing has known withheld truth and the verdict is
      `not-measurable`.
    - `AnomalyResult`: with `labels` (`series_id`, `timestamp`, `is_injected_anomaly`) the
      `top_k_recall` of the raw-score ranking on `channel` is reported with the verdict
      `sample-sanity`; without labels the verdict is `not-measurable`.
    Everything the metric helpers would raise is raised here unchanged.
    """
    base: dict[str, Any] = {"sample_kind": sample_kind, **_identity_block(model)}
    if isinstance(result, EmbeddingResult):
        return _not_measurable(
            {
                **base,
                "task": "embedding",
                "score_semantics": (
                    f"{result.d_model}-dimensional {result.reduction}-pooled window "
                    "representations; not predictions, no correctness metric exists"
                ),
                "n_windows": len(result.window_ids),
                "embedding_shape": list(result.embeddings.shape),
            },
            "embeddings are representations, not predictions",
            "a downstream labelled task (e.g. window classes or retrieval pairs) scored by a "
            "probe or nearest-neighbour evaluation on held-out labels",
        )
    if isinstance(result, ReconstructionResult):
        report_base = {
            **base,
            "task": "imputation",
            "score_semantics": (
                "reconstructed values at deliberately hidden, source-observed points; "
                "source-missing points have no truth"
            ),
            "n_windows": len(result.window_ids),
            "masked_point_fraction": float(result.masked_point_fraction),
            "model_masked_point_fraction": float(result.model_masked_point_fraction),
            "masked_patch_fraction": float(result.masked_patch_fraction),
        }
        requested = result.requested_visible_mask
        if requested is None or not np.any(requested == 0):
            return _not_measurable(
                report_base,
                "no deliberately hidden source-observed points exist in this result",
                "an artificial mask over source-observed positions (pass mask= to impute()) so "
                "masked_point_metrics can score the reconstruction against withheld truth",
            )
        metrics = masked_point_metrics(result)
        estimation = "single deterministic artificial patch holdout on one sample; no dispersion"
        baselines = []
        if baseline is not None:
            baselines.append(
                {
                    "id": "linear_interpolation",
                    "metrics": [
                        {"id": "linear_interpolation", "metric": key, "value": float(value)}
                        for key, value in baseline.items()
                    ],
                }
            )
        return {
            **report_base,
            "n_scored": metrics.n,
            "metrics": [
                {
                    "id": "masked_point_metrics",
                    "metric": "mae",
                    "value": metrics.mae,
                    "estimation": estimation,
                },
                {
                    "id": "masked_point_metrics",
                    "metric": "rmse",
                    "value": metrics.rmse,
                    "estimation": estimation,
                },
            ],
            "baselines": baselines,
            "verdict": "sample-sanity",
            "reason": (
                f"{metrics.n} deliberately hidden points from one artificial holdout; "
                "not a benchmark"
            ),
            "needs": (
                "many artificial holdouts over held-out series of the deployment domain, compared "
                "against domain baselines, for any generalisable imputation claim"
            ),
        }
    if isinstance(result, AnomalyResult):
        report_base = {
            **base,
            "task": "anomaly-scoring",
            "score_semantics": (
                f"raw {result.loss} reconstruction residual per scored position; higher means "
                "more anomaly evidence; no threshold is shipped"
            ),
            "n_windows": len(result.window_ids),
            "scored_point_fraction": float(result.scored_point_fraction),
            "threshold_policy": result.threshold_policy,
        }
        if labels is None:
            return _not_measurable(
                report_base,
                "no anomaly labels were supplied for the scored positions",
                "labelled anomalies (series_id, timestamp, is_injected_anomaly) for the scored "
                "positions so top_k_recall can be computed on the raw-score ranking, or an "
                "operator-calibrated threshold from the deployment domain",
            )
        recall = top_k_recall(result.to_frame(), labels, channel=channel)
        return {
            **report_base,
            "metrics": [
                {
                    "id": "top_k_recall",
                    "k": recall["k"],
                    "channel": recall["channel"],
                    "value": recall["value"],
                    "estimation": "single labelled sample; no dispersion estimate",
                }
            ],
            "baselines": [],
            "verdict": "sample-sanity",
            "reason": f"{recall['k']} labelled positions on one sample; not a benchmark",
            "needs": (
                "labelled anomalies from the deployment domain scored over many series, plus an "
                "operator-chosen threshold, for any generalisable detection claim"
            ),
        }
    raise TypeError(f"unsupported result type {type(result).__name__}")

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `3`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `9fea447e740e…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `load_moment(task="embedding", weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "moment-1-base",
  "modelId": "AutonLab/MOMENT-1-base",
  "revision": "9fea447e740eb968a9e8d80c7562ae122bdb5dde",
  "files": [
    {
      "path": "README.md",
      "bytes": 6349,
      "sha256": "6640901cb08586c3ba2e440b72de58f13aa1415c5b6d6454bbf7f30cfc503e88"
    },
    {
      "path": "config.json",
      "bytes": 949,
      "sha256": "f1c66c2bb845229c0ed27a1600dbcc956b85ab21f9e5fd8a1663e6641bed7755"
    },
    {
      "path": "model.safetensors",
      "bytes": 453940120,
      "sha256": "1a436826ffe618273ec62b9656dc4cab8edc470364f104e90542a4ebc14fb825"
    }
  ],
  "totalBytes": 453947418
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = load_moment(task="embedding", weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Referenced corpus, validation and volunteer-level split

`fetch_corpus` downloads the pinned archive (or reads it from the cache) and refuses a byte-size or SHA-256 mismatch before it is opened; `read_corpus` cuts the 179 windows out of the raw sensor files in memory — per volunteer and activity, the centre 512 samples of the first labelled segment at least 512 samples long (one volunteer has no downstairs segment that long) — each a `{id, x, label}` record with its volunteer, experiment and sample offset. `build_sample_dataset` draws 18 training, 6 validation and 6 test **volunteers** by a seeded shuffle; `validate_dataset` then checks every record against the contract, `check_split_disjoint` asserts no window (by sample digest) and no volunteer appears in two splits, `user_summary` reports volunteers and label counts per split, and the training records table is written to `outputs/moment_classification_train.csv` in the shape BYOD expects.

Look for: 179 windows, six classes of 29–30, splits 107 / 36 / 36 with 18 / 6 / 6 volunteers, three digests, and four refusal probes — a duplicate id, a window of the wrong length, a non-finite window and a single-class dataset — each rejected before `torch` does anything. Success here means the corpus was fetched, verified, cut and split with no leakage; about a minute on the first run for the download.

In [ ]:
import hashlib
import io
import json
import time

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
t0 = time.perf_counter()
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    records = load_byod_dataset(byod_path)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_count = {'byod': len(records)}
else:
    corpus = read_corpus(fetch_corpus(cache_dir='weights/hapt'))
    raw_count = {'windows': len(corpus), 'volunteers': len({r['user'] for r in corpus}), 'channels': list(CHANNELS)}
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME} ({CORPUS_RELEASE}; {CORPUS_LICENSE})'
fetch_seconds = round(time.perf_counter() - t0, 1)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
classes = class_names(train_records)
disjoint = check_split_disjoint(splits)
summary = user_summary(splits)
write_dataset_csv(train_records, 'outputs/moment_classification_train.csv')
print({'data_source': data_source, 'raw': raw_count, 'splits': disjoint, 'classes': classes, 'user_summary': summary, 'fetch_seconds': fetch_seconds, 'corpus_bytes': CORPUS_BYTES})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'users': manifest['users'], 'label_counts': manifest['label_counts'], 'n_channels': manifest['n_channels'], 'value_range': manifest['value_range'], 'digest': manifest['digest'][:16] + '...'}})
example = train_records[0]
print({'example': {k: example[k] for k in ('id', 'label', 'user', 'experiment', 'first_sample', 'source') if k in example}, 'shape': example['x'].shape})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'wrong window length': [{**train_records[0], 'x': train_records[0]['x'][:, :-1]}, *train_records[1:8]],
    'non-finite window': [{**train_records[0], 'x': train_records[0]['x'] * np.nan}, *train_records[1:8]],
    'single class': [{**r, 'label': 'motion'} for r in train_records[:8]],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Embed through the inference contract

Before any adaptation, the inference contract is exercised as it always was, on three test windows. `records_to_long_frame` renders them as the `series_id, timestamp, channel, value` table every inference tutorial starts from; `validate_inputs` runs the package's `validate_long_frame` → `to_windows` path and returns an input manifest, while a deliberately broken frame — a duplicated (series, channel, timestamp) row — is validated too and its rejection recorded as a finding. `embed` returns one unit-norm 768-d vector per window, in window order, with `d_model`, `reduction` and the missingness fractions (0 here: the windows are complete); `evaluation_report` stays `not-measurable`, as it must for a batch of vectors, and `build_provenance` records the model, runtime and inference blocks. Two cosine similarities are printed — a same-activity pair and a cross-activity pair — as a qualitative look at the representation before any metric is read; **cosine is a similarity, not a score**, and a vector carries no label. Success here means the three stages ran and the four sanity checks are `True`.

In [ ]:
probe_records = test_records[:3]
config = MomentConfig(task='embedding')
print({'ceilings': {'max_windows': config.limits.max_windows, 'max_channels': config.limits.max_channels, 'max_rows': config.limits.max_rows}, 'contract': {'sequence_length': config.sequence_length, 'patch_length': config.patch_length, 'd_model': pipe.identity.d_model_effective, 'ENCODER_BLOCKS': ENCODER_BLOCKS, 'task': pipe.identity.task}})
frame = records_to_long_frame(probe_records)
input_manifest = validate_inputs(frame, config, names=[r['id'] for r in probe_records])
broken = pd.concat([frame, frame.iloc[:1]], ignore_index=True)
try:
    validate_inputs(broken, config)
except ValidationError as exc:
    input_manifest['findings'].append({'input': 'duplicate-row-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/moment_classification_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False, default=str)
report, normalized = validate_long_frame(frame, config)
windows = to_windows(normalized, config, report=report, frame=normalized)
started = time.perf_counter()
result = embed(windows, pipe, warmup=False)
embed_seconds = round(time.perf_counter() - started, 3)
vectors = np.asarray(result.embeddings, dtype=np.float32)
vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
checks = {
    'one_vector_per_window': vectors.shape == (3, result.d_model) and result.d_model == pipe.identity.d_model_effective,
    'window_order': list(result.series_ids) == list(windows.series_ids),
    'complete_windows': result.masked_point_fraction == 0.0 and not any(result.padded),
    'all_values_finite': bool(np.isfinite(vectors).all()),
}
if not all(checks.values()):
    raise RuntimeError(f'embed output failed a sanity check: {checks}')
batch_report = evaluation_report(result, model=pipe, sample_kind='three HAPT test windows' if not USE_BYOD else 'three BYOD test windows')
provenance = build_provenance(pipe, windows, result)
ordered = {sid: next(r for r in probe_records if r['id'] == sid) for sid in result.series_ids}
same = next((r for r in test_records[3:] if r['label'] == ordered[result.series_ids[0]]['label']), test_records[3])
other = next((r for r in test_records[3:] if r['label'] != ordered[result.series_ids[0]]['label']), test_records[4])
pair_vectors, _pair = features([same, other], pipe)
print({'probe_ids': list(result.series_ids), 'probe_labels': [ordered[s]['label'] for s in result.series_ids], 'seconds': embed_seconds, 'device': pipe.identity.device, 'checks': checks, 'findings': len(input_manifest['findings']), 'batch_report_verdict': batch_report['verdict'], 'missingness_visible_to_model': result.missingness_visible_to_model})
print({'cosine_same_activity': round(float(vectors[0] @ pair_vectors[0]), 4), 'cosine_other_activity': round(float(vectors[0] @ pair_vectors[1]), 4), 'note': 'one pair each; a similarity, not a score'})

## 6. The majority floor, the k-NN baseline and the frozen policy

Three numbers frame the adaptation, all on the 36 test windows. The **majority floor** predicts the most frequent training activity for every window — 1 / 6 here, since the sample is balanced. The **cosine 5-NN vote** labels each test window by its five nearest training windows in the frozen embedding space: what the representation gives with no training at all. The **frozen policy** is `adapt` with `trainable_blocks=0`: a linear head over the frozen, L2-normalised pooled embeddings, trained full-batch with AdamW for `PROBE_STEPS` steps — the linear probe — scored on validation as epoch 0 of its history and then on the test split by `evaluate` (accuracy, macro-F1, per-class recall, confusion). The build record saw the k-NN vote at 72.2 % and the probe at 72.2 %; read the per-class recall: the three walking activities are separated almost perfectly and the three static postures are confused with each other, because MOMENT instance-normalises every window and the postures differ mainly in the constant gravity component that normalisation removes. Success here means the probe beats the floor; about half a minute on CPU.

In [ ]:
PROBE_STEPS = 300  # @param {type:"integer"}
PROBE_LR = 0.01  # @param {type:"number"}

def brief(m):
    return {'accuracy': round(m['accuracy'], 4), 'macro_f1': round(m['macro_f1'], 4), 'n': m['n']}

floor = majority_baseline([r['label'] for r in train_records], [r['label'] for r in test_records], classes)
print({'majority_floor': brief(floor), 'baseline': floor['baseline']})
t0 = time.perf_counter()
baseline_knn = knn_baseline(pipe, train_records, test_records, k=5)
print({'knn_baseline': brief(baseline_knn), 'baseline': baseline_knn['baseline'], 'seconds': round(time.perf_counter() - t0, 1)})
t0 = time.perf_counter()
probe_adapter = adapt(pipe, train_records, val_records, probe_steps=PROBE_STEPS, probe_lr=PROBE_LR, trainable_blocks=0)
frozen_test = evaluate(pipe, probe_adapter, test_records)
print({'frozen_policy': probe_adapter.policy, 'probe_final_loss': round(probe_adapter.config['probe_final_loss'], 4), 'validation': probe_adapter.history[0]['val'], 'seconds': round(time.perf_counter() - t0, 1)})
print({'frozen_policy_test': brief(frozen_test), 'log_loss': round(frozen_test['log_loss'], 4), 'verdict': frozen_test['verdict'], 'per_class_recall': {c: round(v['recall'], 2) for c, v in frozen_test['per_class'].items()}})
print({'definitions': frozen_test['definitions']})
assert frozen_test['accuracy'] > floor['accuracy'] and probe_adapter.policy == POLICY_FROZEN

## 7. The unfrozen policy: a bounded unfreeze selected against the probe

`adapt` with `TRAINABLE_BLOCKS` > 0 first retrains the linear probe on the frozen embeddings (epoch 0 of the history, the frozen policy), then unfreezes the last `TRAINABLE_BLOCKS` T5 encoder blocks — two by default, 14,158,848 of 109,635,456 parameters; the patch embedding, the earlier blocks and the final norm stay frozen — and trains them with the head end to end on the windows for `EPOCHS` epochs (AdamW at `LEARNING_RATE`, weight decay 0.01, gradient clipping 1.0, seeded shuffling, no augmentation). Every epoch is scored on validation, and the epoch with the **lowest validation log-loss** is kept — epoch 0, the probe, competes on equal terms, so the selected policy can be either; the encoder inside `pipe` is modified in place only when the unfreeze wins. Accuracy and macro-F1 are printed beside the loss at every epoch.

Watch the validation log-loss: the build record's sweep on this sample — two blocks at 3e-4 for three epochs went 0.746 (probe) → 0.703 → 0.751 → 0.605 and was selected at epoch 3, for 77.8 % held-out accuracy against the probe's 72.2 % and the static postures partly recovered; at 1e-4 epoch 2 was selected for 75.0 %; at 3e-5 no epoch beat the probe's validation log-loss and the probe was kept. Success here means the ladder ran to its last epoch and named a selected policy and a `best_epoch`; about three minutes on CPU.

In [ ]:
EPOCHS = 3  # @param {type:"integer"}
LEARNING_RATE = 3e-4  # @param {type:"number"}
BATCH_SIZE = 8  # @param {type:"integer"}
TRAINABLE_BLOCKS = 2  # @param {type:"integer"}

def report_epoch(entry):
    row = {'epoch': entry['epoch'], 'stage': entry['stage'], 'train_loss': round(entry['train_loss'], 4)}
    if entry.get('val'):
        row['val_log_loss'] = round(entry['val']['log_loss'], 4)
        row['val_accuracy'] = round(entry['val']['accuracy'], 4)
        row['val_macro_f1'] = round(entry['val']['macro_f1'], 4)
    print(row)

t0 = time.perf_counter()
adapter = adapt(pipe, train_records, val_records, probe_steps=PROBE_STEPS, probe_lr=PROBE_LR, trainable_blocks=TRAINABLE_BLOCKS, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, progress=report_epoch)
adapt_seconds = round(time.perf_counter() - t0, 1)
report_epoch(adapter.history[0])
print({'selected_policy': adapter.policy, 'best_epoch': adapter.config['best_epoch'], 'selection': adapter.config['selection'], 'trainable_head': adapter.config['n_trainable_head'], 'trainable_blocks': adapter.config['n_trainable_blocks'], 'total_parameters': adapter.config['n_total'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test split was never used for training or policy selection, and no volunteer in it appears in the training or validation splits. The selected model is scored exactly as the frozen policy was in Section 6, and the four rows are put side by side: majority floor, k-NN vote, frozen policy, selected policy. Read the policy first: if validation kept the probe, the last two rows are the same model; if it chose the unfreeze, the delta is what the unfreeze bought on 36 windows — the build record: 77.8 % / macro-F1 0.776 against the probe's 72.2 % / 0.715, two windows, with laying and sitting recall rising from 0.33 to 0.50 while the three walking activities stayed at 1.0. The cell asserts the selected model beats the majority floor; it does **not** assert a gain over the probe, because that is the question, not the answer. 36 windows from six volunteers of one seeded split give no dispersion estimate — one window is about 2.8 points of accuracy. Success here means the comparison and the evaluation report were written.

In [ ]:
adapted_test = evaluate(pipe, adapter, test_records)
adapted_val = evaluate(pipe, adapter, val_records)
comparison = {
    metric: {'majority_floor': round(floor[metric], 4), 'knn5': round(baseline_knn[metric], 4), 'frozen_policy': round(frozen_test[metric], 4), 'selected_policy': round(adapted_test[metric], 4)}
    for metric in ('accuracy', 'macro_f1')
}
comparison['log_loss'] = {'frozen_policy': round(frozen_test['log_loss'], 4), 'selected_policy': round(adapted_test['log_loss'], 4)}
comparison['per_class_recall'] = {c: {'knn5': round(baseline_knn['per_class'][c]['recall'], 2), 'frozen': round(frozen_test['per_class'][c]['recall'], 2), 'selected': round(adapted_test['per_class'][c]['recall'], 2)} for c in classes}
comparison['delta_vs_frozen'] = {metric: round(adapted_test[metric] - frozen_test[metric], 4) for metric in ('accuracy', 'macro_f1')}
comparison['selected_policy'] = adapter.policy
for metric, row in comparison.items():
    print({metric: row})
print({'confusion_selected': adapted_test['confusion'], 'classes': classes})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'user_summary': summary,
    'classes': classes,
    'batch_report': batch_report,
    'baselines': {'majority_floor': floor, 'knn5': baseline_knn},
    'frozen_policy': {'adaptation': probe_adapter.summary(), 'history': probe_adapter.history, 'test': frozen_test},
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'comparison': comparison,
    'adaptation': adapter.summary(),
    'history': adapter.history,
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/moment_classification_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['accuracy'] > floor['accuracy']
print({'report': 'outputs/moment_classification_evaluation_report.json'})

## 9. Predict before and after, export the adapter and reload it

Six test windows are labelled by `classify` with the selected model and printed beside the frozen policy's predictions (from the probe adapter of Section 6 over a fresh, untouched `load_moment` instance — after an unfreeze the encoder inside `pipe` has moved, so the frozen column needs its own encoder) and the gold activity with the top probability; read the probabilities as the head's softmax, not a calibrated confidence.

`save_artifact` writes the head (`head.weight`, `head.bias`) and, when the unfrozen policy was selected, the trained block tensors — 18.6 KB for the head alone, about 56.7 MB with two trained blocks — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the classes, the selected policy, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `load_artifact` re-verifies the manifest and digest **before** deserialising, rebuilds the head from the manifest's classes, refuses any tensor that is not an encoder-block tensor of the base, and overlays the tensors onto a freshly loaded, digest-verified base — a new object from files, not the in-memory model (VER2). The cell asserts identical probabilities on the six windows and an identical test accuracy (VER4). Success here means the parity assertion passed and the six exports exist.

In [ ]:
import csv
import shutil

show = test_records[:6]
after = classify(pipe, adapter, show)
frozen_pipe = load_moment(task='embedding', weights_dir=WEIGHTS_DIR)
before = classify(frozen_pipe, probe_adapter, show)
gold = {r['id']: r for r in show}
rows = []
for i, rid in enumerate(after['ids']):
    rows.append({'id': rid, 'gold': gold[rid]['label'], 'frozen_label': before['labels'][i], 'frozen_top_probability': round(max(before['probabilities'][i]), 4), 'selected_label': after['labels'][i], 'selected_top_probability': round(max(after['probabilities'][i]), 4), 'user': gold[rid].get('user', ''), 'source': gold[rid].get('source', '')})
    print({k: rows[-1][k] for k in ('id', 'gold', 'frozen_label', 'frozen_top_probability', 'selected_label', 'selected_top_probability')})
print({'selected_policy': after['policy'], 'decision_rule': after['decision_rule'], 'predictions_changed': sum(r['frozen_label'] != r['selected_label'] for r in rows), 'of': len(rows)})
with open('outputs/moment_classification_predictions.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

artifact_dir = Path('outputs/moment_classification_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
save_artifact(pipe, adapter, artifact_dir, metadata={'tutorial': 'moment_classification', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'policy': artifact_manifest['adapter']['policy'], 'classes': artifact_manifest['adapter']['classes'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded_pipe = load_moment(task='embedding', weights_dir=WEIGHTS_DIR)
reloaded = load_artifact(reloaded_pipe, artifact_dir)
reloaded_after = classify(reloaded_pipe, reloaded, show)
reloaded_test = evaluate(reloaded_pipe, reloaded, test_records)
parity = {'probabilities_identical': reloaded_after['probabilities'] == after['probabilities'], 'accuracy_in_memory': round(adapted_test['accuracy'], 6), 'accuracy_reloaded': round(reloaded_test['accuracy'], 6), 'classes_identical': reloaded.classes == adapter.classes}
print({'reload_parity': parity, 'reloaded_policy': reloaded.policy, 'reloaded_best_epoch': reloaded.config['best_epoch']})
assert parity['probabilities_identical'] and parity['classes_identical'] and abs(adapted_test['accuracy'] - reloaded_test['accuracy']) < 1e-9

provenance['data'] = {'kind': 'byod' if USE_BYOD else 'referenced-corpus', 'name': data_source, 'corpus_sha256': CORPUS_SHA256, 'corpus_bytes': CORPUS_BYTES, 'dataset_digests': evaluation_report_payload['dataset_digests']}
provenance['notebook_source'] = NOTEBOOK_SOURCE
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes'], 'fetched_this_run': fetched, 'weight_file': pipe.identity.weight_file_loaded, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': pipe.identity.weights_sha256},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'release': CORPUS_RELEASE, 'url': CORPUS_URL, 'bytes': CORPUS_BYTES, 'sha256': CORPUS_SHA256, 'license': CORPUS_LICENSE, 'windows': raw_count, 'activities': list(ACTIVITIES.values())},
    'inference_contract': {'input_manifest': input_manifest, 'sanity_checks': checks, 'probe_ids': list(result.series_ids), 'batch_report': batch_report, 'seconds': embed_seconds},
    'provenance': provenance,
    'comparison': comparison,
    'predictions_before_after': rows,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors']), 'policy': artifact_manifest['adapter']['policy']},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'pandas': pandas.__version__, 'numpy': numpy.__version__, 'device': pipe.identity.device, 'dtype': pipe.identity.dtype},
}
with open('outputs/moment_classification_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False, default=str)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The frozen MOMENT embeddings already separate the three walking activities perfectly and put a cosine 5-NN vote and a linear probe at 72.2 % on 36 held-out windows against a majority floor of one in six, and a bounded unfreeze of the last two encoder blocks on 107 training windows — selected against the probe by validation log-loss — was selected at its last epoch and reached 77.8 % in the build record, two windows better, by partly recovering the static postures. That is the claim and the finding: the adaptation contract runs both policies end to end on a real labelled corpus through the package's own validation and windowing path, chooses between them on validation rather than by assumption, and reports the answer against a floor and a no-training baseline rather than in isolation.

The test split is 36 windows from six volunteers of one seeded split of one small corpus with no dispersion estimate — one window is about 2.8 points of accuracy, so a three-point delta is noise. The representation finding is the more useful one: MOMENT instance-normalises every window before patching, so the constant gravity component that tells sitting from standing from laying never reaches the encoder — the walking activities, which differ in dynamics, separate almost perfectly, while the static postures collapse into each other under both policies; a head cannot recover information the normalisation removed. Accuracy and macro-F1 say whether the gold activity is predicted, not whether the embeddings are good for any other task; the head's softmax is not a calibrated confidence. When the unfrozen policy is selected it changes the last blocks, which every window shares, so `embed` returns different vectors after it — cosines are not comparable across the frozen and adapted encoders, and the artifact records which policy won.

Three things to carry to real data. **Floors first:** the majority floor and the k-NN vote on *your* windows are the numbers to read before any trained head's — if the probe barely beats k-NN, the representation already carries the task. **Leakage:** split by person, device or session (the contract splits by `user` / `group`, never by window). **Normalisation:** if your classes differ by level or offset rather than by shape, carry that information as an extra channel or a side feature — the encoder will not see it.

Successful execution proves that the recorded repository revision's package, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real labelled corpus, validate the demonstrated dataset contract without leakage, execute the inference contract on real windows, a linear probe and a bounded unfreeze with validation-based policy selection, evaluate by accuracy and macro-F1 against a floor and a k-NN baseline on an independent split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, representation quality on any other task, a usable acceptance threshold, or production fitness.

**Optional experiments (they do not affect the default path):** set `LEARNING_RATE = 3e-5` and watch the probe win on validation (the build record: no epoch beat it); set `LEARNING_RATE = 1e-4` (epoch 2 selected, 75.0 %); set `TRAINABLE_BLOCKS = 1` or `4`; set `EPOCHS = 6` and watch whether validation log-loss keeps falling or turns; drop the gyroscope channels in `read_corpus` and read what the accelerometer alone carries; or bring your own labelled windows through BYOD and read the k-NN baseline before either policy.

## References

- Repository README: https://github.com/kurtvalcorza/moment-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/moment-pipeline/blob/main/MODEL_CARD.md
- Upstream model: https://huggingface.co/AutonLab/MOMENT-1-base
- Upstream code: https://github.com/moment-timeseries-foundation-model/moment
- MOMENT: A Family of Open Time-series Foundation Models (Goswami et al., 2024): https://arxiv.org/abs/2402.03885
- Reyes-Ortiz, Oneto, Samà, Parra, Anguita. Transition-Aware Human Activity Recognition Using Smartphones. Neurocomputing 171 (2016) — the HAPT dataset, UCI Machine Learning Repository id 341, CC BY 4.0: https://archive.ics.uci.edu/dataset/341
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)